<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_07_model_analysis/seq2one/stage_07_02a_transformer_seq2one_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_07_02a - SEQ2ONE - Transformer - Tuning**

El Transformer es un modelo basado en self-attention que procesa secuencias completas en paralelo, sin recurrencia. En el esquema many-to-one, el modelo recibe una ventana temporal de múltiples pasos (many) y produce una única salida agregada (one), típicamente usando el embedding del último token o un pooling sobre la secuencia.

Su ventaja clave es capturar dependencias de largo alcance de forma eficiente, con alta escalabilidad y estabilidad en el entrenamiento frente a RNN/LSTM.


In [1]:
window_sizes = [180]
targets = ['delta_60']
splits = ['train', 'valid', 'test']

# **BLOQUE DE EJECUCIÓN COMPLETO**

## **1. Imports + paths**

In [2]:
# Permite anotaciones modernas en Python < 3.11.
from __future__ import annotations

import os
import json
import time
import random
from pathlib import Path
import numpy as np

import numpy as np
from typing import Any, Dict, List, Iterable, Tuple
# Si usa sklearn en algún modelo/baseline
# from sklearn.linear_model import Ridge

# Dataclass: contenedor simple e inmutable para configuración.
from dataclasses import dataclass
# Joblib para cargar scalers .pkl (sklearn).
import joblib

from pathlib import Path
import sys
import importlib

## **2. Acceso a drive**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# --------------------------------------------------
# Ruta base en Google Drive
# --------------------------------------------------
# Ajuste si su estructura cambia
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

Mounted at /content/drive


## **3. Rutas de ventanas seq2one y scalers**

In [4]:
from pathlib import Path
import os

WINDOWS_SEQ2ONE_DIR = Path(
    os.environ.get("WINDOWS_SEQ2ONE_DIR", "data/windows/seq2one/")
)

SCALERS_DIR = Path(
    os.environ.get("SCALERS_DIR", "data/scaled/")
)

In [5]:
windows_paths = {}

for w in window_sizes:
    windows_paths[w] = {}

    for t in targets:
        windows_paths[w][t] = {}

        for s in splits:
            path = (
                DRIVE_DIR
                / WINDOWS_SEQ2ONE_DIR
                / f"L{w}"
                / f"windows_{t}_{s}.npz"
            )

            windows_paths[w][t][s] = path

#display(windows_paths)

#Como llamarlo:
#path_train_L60_delta = windows_paths[60]['delta_90']['train']
#print(path_train_L60_delta)

In [6]:
scalers_paths = {}
for t in targets:
  scalers_paths[t] = {}
  path = (
                DRIVE_DIR
                / SCALERS_DIR
                / f"scaler_{t}.pkl"
            )

  scalers_paths[t] = path

display(scalers_paths)

{'delta_60': PosixPath('/content/drive/MyDrive/neural_profit/data/scaled/scaler_delta_60.pkl')}

## **4. Reproducibilidad**

In [7]:
#def set_seeds(seed: int = 42) -> None:
#    random.seed(seed)
#    np.random.seed(seed)
#    os.environ["PYTHONHASHSEED"] = str(seed)
#
#set_seeds(42)

## **5. Importar métricas comunes desde .py**

In [8]:
# 1) Define el root del proyecto (DEBE existir en esta misma celda)
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

# 2) Agrega el root al PYTHONPATH (antes de importar)
if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))  # insert(0) para priorizarlo

# 3) Asegura que metrics sea paquete Python
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

# 4) Invalida cachés de import (importante en Colab)
importlib.invalidate_caches()

# 5) Diagnóstico (le muestra qué ve Python)
#print("DRIVE_DIR:", DRIVE_DIR)
#print("sys.path[0]:", sys.path[0])
#print("metrics exists:", (DRIVE_DIR / "metrics").exists())
#print("metrics __init__:", (DRIVE_DIR / "metrics" / "__init__.py").exists())

# 6) Imports reales
from metrics.seq2one_metrics import compute_seq2one_metrics
#from metrics.opportunity_filter_metrics import compute_opportunity_filter_metrics

print("OK - imports metrics.*")


OK - imports metrics.*


In [9]:
#print(compute_seq2one_metrics.__doc__)

## **6. Carga de data windows**

In [10]:
def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar.
    """

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # Usar contexto para cerrar correctamente el archivo
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")
        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # Opcional pero recomendable: copiar a memoria
        X = X.copy()
        y = y.copy()

    return X, y

In [11]:
# --------------------------------------------------
# Función común: carga scaler .pkl (sklearn)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib (ej. StandardScaler/MinMaxScaler).
    """
    # Verifica existencia del archivo.
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # Carga el objeto scaler.
    scaler = joblib.load(path)

    # Devuelve scaler (tipo genérico).
    return scaler

In [12]:
from typing import Any, Dict, Mapping
from pathlib import Path

def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scalers_path: Mapping[str, Path],
    splits: tuple[str, ...] = ("train", "valid", "test"),
) -> Dict[str, Any]:
    """
    Carga X/y para los splits solicitados y el scaler correspondiente
    a un (window_size, target).

    windows_paths[L][target][split] -> Path (.npz con X,y)
    scalers_path[target] -> Path (scaler)
    """

    # --------------------------
    # 1) Validaciones base
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    if target not in scalers_path:
        raise KeyError(f"target='{target}' no existe en scalers_path")

    valid_splits = {"train", "valid", "test"}
    splits_set = set(splits)
    unknown = splits_set - valid_splits
    if unknown:
        raise ValueError(f"splits inválidos: {sorted(unknown)}. Usar {sorted(valid_splits)}")

    # Validar que existan los splits solicitados para ese target
    available = set(windows_paths[window_size][target].keys())
    missing = splits_set - available
    if missing:
        raise KeyError(
            f"Faltan splits {sorted(missing)} en windows_paths[{window_size}]['{target}']. "
            f"Disponibles: {sorted(available)}"
        )

    # --------------------------
    # 2) Paths (solo los necesarios)
    # --------------------------
    split_paths: Dict[str, Path] = {sp: windows_paths[window_size][target][sp] for sp in splits}
    scaler_path = scalers_path[target]

    # --------------------------
    # 3) Carga por split
    # --------------------------
    out_splits: Dict[str, Dict[str, Any]] = {}
    out_paths: Dict[str, str] = {}

    for sp, p in split_paths.items():
        X, y = load_npz_windows(p)
        out_splits[sp] = {"X": X, "y": y}
        out_paths[sp] = str(p)

    scaler = load_scaler(scaler_path)
    out_paths["scaler"] = str(scaler_path)

    # --------------------------
    # 4) Inferir horizonte (robusto)
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception as e:
        raise ValueError(f"No se pudo inferir horizon desde target='{target}'. Esperado sufijo '_<int>'") from e

    # --------------------------
    # 5) Retorno
    # --------------------------
    out: Dict[str, Any] = {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": out_paths,
        "scaler": scaler,
    }
    out.update(out_splits)

    return out

In [13]:
import numpy as np

def maybe_flatten_X(X: np.ndarray, *, flatten: bool) -> np.ndarray:
    """
    Si flatten=True y X es 3D (N,L,F) -> (N, L*F)
    Si flatten=False -> retorna X tal cual.
    """
    if not flatten:
        return X
    if X.ndim == 3:
        N, L, F = X.shape
        return X.reshape(N, L * F)
    if X.ndim == 2:
        return X
    raise ValueError(f"X debe ser 2D o 3D, recibí shape={X.shape}")

In [14]:
def create_bundles(
    window_size,
    targets: list,
    windows_paths=windows_paths,
    scalers_paths=scalers_paths,
    *,
    flatten_X: bool = False,
    splits: tuple[str, ...] = ("train", "valid", "test"),
    verbose_shapes: bool = True,
    ):
    bundles = []
    for t in targets:
        b = load_windows_and_scaler(
            window_size=window_size,
            target=t,
            windows_paths=windows_paths,
            scalers_path=scalers_paths,
            splits=splits,
        )

        if flatten_X:
            for sp in splits:
                b[sp]["X"] = maybe_flatten_X(b[sp]["X"], flatten=True)

        bundles.append(b)

    if verbose_shapes:
        for b in bundles:
            for sp in splits:
                print(f"H{b['horizon']} {sp.capitalize():<5}:", b[sp]["X"].shape, b[sp]["y"].shape)
            print(f"Scaler H{b['horizon']}:", type(b["scaler"]).__name__)

    return tuple(bundles)

NOTA IMPORTANTE: COMO ACCEDER A LAS VENTANAS

Para el horizonte: `h`

  - Train
    - `X`: `bundle_h["train"]["X"]`
    - `y`: `bundle_h["train"]["y"]`

  - Valid
    - `X`: `bundle_h["valid"]["X"]`
    - `y`:`bundle_h["valid"]["y"]`

  - Test
    - `X`: `bundle_h["test"]["X"]`
    - `y`: `bundle_h["test"]["y"]`

  - Scaler
    - `bundle_h["scaler"]`

## **7. Sanity Check**

In [15]:
from __future__ import annotations

from typing import Any, Dict, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_float_array(a: Any, *, name: str) -> np.ndarray:
    """Convierte a np.ndarray float64 y valida finitud."""
    arr = np.asarray(a, dtype=np.float64)
    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")
    if not np.isfinite(arr).all():
        raise ValueError(f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}")
    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).
    Acepta: (n,), (n,1). Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y
    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)
    raise ValueError(f"{name} shape inválido para seq2one. Se esperaba (n,) o (n,1). Recibido {y.shape}")


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiera (seq_len, n_features, mode) desde X.
    mode:
      - "3d": X=(n, seq_len, n_features)
      - "2d": X=(n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"
    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"
    raise ValueError(f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}")

In [16]:
# ============================================================
# 2) Sanity check principal (seq2one)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como "d_flat" esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado). Ej: 60*20=1200.
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len) (caso raro), permite tomar y[:, -1].
        Por defecto False (recomendado).
    """
    X = _as_float_array(X, name=f"X[{split_name}]")
    y = _as_float_array(y, name=f"y[{split_name}]")

    # Normalizar y
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        # caso tolerante: y=(n,seq_len) -> tomar último
        y = y[:, -1]
    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # Inferir modo y dimensiones de X
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # Validaciones básicas n_samples
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # Validación de shapes según modo
    if mode == "3d":
        # expected_seq_len / expected_n_features
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, recibido={seq_len}. X.shape={X.shape}"
            )
        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        # Si expected_flat_dim está, valida contra eso
        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, recibido={d_flat}. X.shape={X.shape}"
            )

        # Si no hay expected_flat_dim pero sí expected_seq_len, úselo como d_flat esperado
        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, recibido={d_flat}. X.shape={X.shape}"
            )

        # expected_n_features no aplica en 2D
        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim (ej: 1200) o pase X en 3D."
            )

    # Validación extra: varianza de y (para detectar targets constantes)
    y_std = float(np.std(y))
    if verbose:
        info = {
            "split": split_name,
            "X_shape": tuple(X.shape),
            "y_shape": tuple(y.shape),
            "mode": mode,
            "seq_len": seq_len if mode == "3d" else None,
            "n_features": n_features if mode == "3d" else None,
            "flat_dim": int(X.shape[1]) if mode == "2d" else None,
            "y_mean": float(np.mean(y)),
            "y_std": y_std,
            "y_min": float(np.min(y)),
            "y_max": float(np.max(y)),
        }
        print(
            f"[sanity_check_seq2one] {split_name} | X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | y_std={y_std:.6f}"
        )

    return {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "y_mean": float(np.mean(y)),
        "y_std": float(np.std(y)),
        "y_min": float(np.min(y)),
        "y_max": float(np.max(y)),
    }

In [17]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "horizon": 60,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # Setear esperados desde TRAIN si no se dieron
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        # En 3D no hace falta expected_flat_dim
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        # Para evitar confusión, no usamos expected_seq_len/n_features en 2D
        expected_seq_len = expected_seq_len  # puede quedar None
        expected_n_features = None

    # Ejecutar checks
    out_tr = sanity_check_seq2one(
        X_tr, y_tr, f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_va = sanity_check_seq2one(
        X_va, y_va, f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )
    out_te = sanity_check_seq2one(
        X_te, y_te, f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    h = bundle.get("horizon", "NA")
    if verbose:
        print(f"OK {tag} (h={h})")

    return {"train": out_tr, "valid": out_va, "test": out_te, "horizon": h}


def run_sanity_checks_all_horizons_seq2one(
    bundle_60: Dict[str, Any],
    bundle_90: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Corre sanity checks para ambos horizontes (ej: 60 y 90)."""
    out_60 = run_sanity_checks_for_bundle_seq2one(bundle_60, tag="h60", verbose=verbose)
    out_90 = run_sanity_checks_for_bundle_seq2one(bundle_90, tag="h90", verbose=verbose)
    return {"h60": out_60, "h90": out_90}

In [18]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **8. Métricas ML**


In [19]:
import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    horizon: int,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte un dict de métricas seq2one en una fila de DataFrame.
    """

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon_min": horizon,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics.get("R2"),
        "DA": metrics.get("DA"),
    }])

## **9. Gestión de dataset de métricas**

In [20]:
def load_seq2one_metrics_if_exists(
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning",
) -> pd.DataFrame:
    path = Path(base_dir) / f"seq2one_{name}_metrics.parquet"
    if path.exists():
        return pd.read_parquet(path)
    return pd.DataFrame()

In [21]:
from pathlib import Path
import pandas as pd

def save_seq2one_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning",
) -> Path:
    """
    Guarda un DataFrame de métricas seq2one en formato Parquet.

    Parámetros
    ----------
    df_metrics : pd.DataFrame
        DataFrame con métricas (una fila por modelo/horizonte/split).
    name : str
        Nombre identificador del archivo (ej: 'naive_valid', 'ridge_valid').
        No incluir extensión.
    base_dir : str
        Directorio base donde se almacenan todas las métricas seq2one.

    Retorna
    -------
    out_path : Path
        Ruta completa del archivo guardado.
    """
    out_dir = Path(base_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / f"seq2one_{name}_metrics.parquet"
    df_metrics.to_parquet(out_path, index=False)

    print(f"[OK] Métricas guardadas en: {out_path}")
    return out_path

In [22]:
import gc, torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

gc.collect()
torch.cuda.empty_cache()

# **DEFINICIÓN DE MODELO**

## **10. Definición del modelo — placeholder**

### **10.1. Modelo Transformer — seq2one**

**Idea básica**

El **Transformer** es un modelo de aprendizaje profundo basado en el
mecanismo de **self-attention**, diseñado para modelar dependencias
temporales **sin recurrencia** y con procesamiento completamente
paralelo de la secuencia de entrada.

En el esquema **many-to-one**, el modelo recibe una **ventana temporal**
de múltiples pasos (many) y produce **un único valor escalar futuro**
(one), asociado al final de la ventana.

A diferencia del MLP, el Transformer **preserva explícitamente la
estructura temporal**, permitiendo que cada instante de la secuencia
atienda a cualquier otro instante según su relevancia para la
predicción final.

Formalmente, el modelo puede representarse como:

$$
\hat{y}_t = g\Big( \text{Pool}\big( \text{TransformerEncoder}(X_t) \big) \Big)
$$

donde:
- $X_t \in \mathbb{R}^{T \times F}$ es la ventana temporal (longitud $T$, $F$ features),
- $\text{TransformerEncoder}(\cdot)$ aplica capas de self-attention y feedforward,
- $\text{Pool}(\cdot)$ es una agregación temporal (último token, mean pooling, etc.),
- $g(\cdot)$ es una capa densa final que produce el target escalar.

---

**Regularización (Transformer)**

**Riesgo:** Alto, debido a la elevada capacidad del modelo.

El Transformer incorpora **regularización parcial de forma intrínseca**,
pero requiere control explícito:

- **Dropout (intrínseco):**
  - Aplicado en self-attention y capas feedforward.
- **Early stopping:**
  - Fundamental para evitar sobreajuste.
- **Dimensión del embedding controlada:**
  - Evita representaciones excesivamente complejas.
- **Número limitado de capas encoder:**
  - 1–3 capas en escenarios de datos financieros.
- **Weight decay (opcional):**
  - Refuerza la estabilidad del entrenamiento.

---

**Por qué el Transformer es relevante en este proyecto**

- Capacidad para capturar:
  - dependencias **de largo alcance**,
  - relaciones temporales no locales.
- Adecuado para:
  - ventanas largas (60–90 minutos),
  - múltiples indicadores técnicos simultáneos.
- Entrenamiento:
  - paralelo y estable,
  - más escalable que LSTM/GRU.

El Transformer representa el **primer modelo plenamente atencional**
del pipeline, sirviendo como referencia frente a arquitecturas
secuenciales (LSTM, GRU) y convolucionales (TCN).

---

**Hiperparámetros iniciales**

Para este stage (sin tuning):

- Número de capas encoder: 1–2
- Dimensión del embedding: moderada (32–64)
- Número de cabezas de atención: 2–4
- Dropout: activado
- Pooling temporal: último token o mean pooling
- Early stopping: activado
- **Sin tuning exhaustivo** (optimización posterior)

El ajuste fino de profundidad, atención y regularización se aborda en
etapas posteriores del proyecto.


**Opción B — Seed fija durante tuning + multi-seed en validación final (recomendado)**

Proceso:

1. Durante el tuning:
    - Usar una seed fija (ej. 42).
    - Buscar las mejores configuraciones según VALID.

2. Después del tuning:
    - Tomar la mejor (o top 3) configuraciones.
    - Evaluarlas con 10–20 seeds independientes.
    - Reportar media, desviación estándar y, si corresponde, tests estadísticos.

Ventajas:
- El tuning es computacionalmente eficiente.
- La robustez se evalúa de forma explícita al final.
- Se evita explosión combinatoria.

Este es el enfoque más equilibrado entre rigor y costo computacional.

**3. Estrategia adoptada para este proyecto**

Dado el nivel de rigurosidad actual del pipeline:

Durante tuning (Optuna / grid search):
- Usar seed fija = 42.
- Comparar configuraciones en VALID.
- Seleccionar top 1 o top 3.

Después del tuning:
- Ejecutar las configuraciones seleccionadas con 20 seeds.
- Elegir la que:
  - Maximice R² medio.
  - Minimice R²_std.
  - Mantenga gap VALID–TEST controlado.

Esto mantiene coherencia con el enfoque multi-seed ya aplicado en la selección de `window_size`.

### **10.2. Imports y “seed” (base reproducible)**

In [23]:
# Paso 1: imports básicos + reproducibilidad (sin tqdm)
import os
import json
import random
from pathlib import Path
from typing import Dict, Any, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [24]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # reproducibilidad (puede bajar performance, pero estable)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


### **10.3. TensorDataset + DataLoader (PyTorch)**

In [25]:
# Generador de Dataloader para tuning con variación por seed.
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

def make_loaders_from_bundle_3d(
    bundle: dict,
    *,
    seq_len: int | None = None,
    n_features: int | None = None,
    batch_size: int = 1024,
    num_workers: int = 2,
) -> dict:
    """
    Crea loaders train/valid/test para TRANSFORMER many-to-one.

    Espera:
      - X: (n, seq_len, n_features)  (ya 3D)
      - y: (n,) o (n,1)  -> (n,1)

    Si seq_len/n_features se pasan, valida consistencia.
    """
    loaders = {}


    import random

    g = torch.Generator()
    g.manual_seed(42)

    def _seed_worker(worker_id):
        worker_seed = 42 + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    for split in ["train", "valid", "test"]:
        X = np.asarray(bundle[split]["X"], dtype=np.float32)
        y = np.asarray(bundle[split]["y"], dtype=np.float32).reshape(-1, 1)

        if X.ndim != 3:
            raise ValueError(
                f"[{split}] Se esperaba X 3D (n, seq_len, n_features). "
                f"Recibido shape={X.shape} (ndim={X.ndim})."
            )

        n, sl, nf = X.shape

        if seq_len is not None and sl != int(seq_len):
            raise ValueError(f"[{split}] seq_len esperado={seq_len}, recibido={sl}. shape={X.shape}")

        if n_features is not None and nf != int(n_features):
            raise ValueError(f"[{split}] n_features esperado={n_features}, recibido={nf}. shape={X.shape}")

        if y.shape[0] != n:
            raise ValueError(f"[{split}] X e y no alinean: X n={n}, y n={y.shape[0]}.")

        ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
        shuffle = (split == "train")
        #
        loaders[split] = DataLoader(
            ds,
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            drop_last=False,
            generator=g if shuffle else None,
            worker_init_fn=_seed_worker if num_workers > 0 else None,
        )

    return loaders

### **10.4. Definición de modelo Transformer (many-to-one)**

In [26]:
import math
import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    """
    Positional Encoding sinusoidal (Vaswani et al.).
    Asume entradas con forma (B, L, D) y agrega información de posición.
    """
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 2048):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        # pe: (max_len, d_model)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (max_len, 1)

        # div_term: (d_model/2,) para índices pares
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)  # pares
        pe[:, 1::2] = torch.cos(position * div_term)  # impares

        # Guardamos como buffer para que:
        # - se mueva con .to(device)
        # - no sea parámetro entrenable
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, d_model)
        L = x.size(1)
        x = x + self.pe[:, :L, :]
        return self.dropout(x)


class TransformerManyToOne(nn.Module):
    """
    Transformer Encoder many-to-one para regresión:
      - Entrada:  (B, L, F)
      - Salida:   (B, 1)

    Sugerencias integradas:
      - Validación d_model % nhead == 0 (evita trials inválidos en tuning).
      - LayerNorm post-proyección de entrada (a menudo estabiliza/regulariza).
      - Pooling configurable ("mean" o "last") para reducir (B, L, D) -> (B, D).

    Nota:
      - No incluye máscaras de padding porque asume L fijo (sin padding).
        Si más adelante usa padding, habrá que pasar src_key_padding_mask al encoder.
    """
    def __init__(
        self,
        *,
        n_features: int,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 2,
        dim_ff: int = 128,
        dropout: float = 0.1,
        pooling: str = "mean",  # "mean" o "last"
        max_len: int = 2048,    # para el positional encoding
    ):
        super().__init__()

        # 1) Validación clave para Optuna/grid: evita combinaciones inválidas
        if d_model % nhead != 0:
            raise ValueError(f"d_model ({d_model}) debe ser múltiplo de nhead ({nhead}).")

        if pooling not in ("mean", "last"):
            raise ValueError(f"pooling inválido: {pooling}. Use 'mean' o 'last'.")

        self.pooling = pooling

        # Proyección de features a dimensión del modelo
        self.in_proj = nn.Linear(n_features, d_model)

        # 2) (Opcional pero recomendado) normalización para estabilizar la entrada
        self.in_norm = nn.LayerNorm(d_model)

        # Positional encoding
        self.pos_enc = PositionalEncoding(d_model=d_model, dropout=dropout, max_len=max_len)

        # Encoder layers
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,   # entrada/salida (B, L, D)
            activation="gelu",
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # Cabeza de regresión
        self.head = nn.Linear(d_model, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, L, F)
        z = self.in_proj(x)   # (B, L, D)
        z = self.in_norm(z)   # (B, L, D)  -> estabiliza escalas
        z = self.pos_enc(z)   # (B, L, D)
        z = self.encoder(z)   # (B, L, D)

        # Pooling: reduce dimensión temporal
        if self.pooling == "last":
            pooled = z[:, -1, :]     # (B, D)
        else:
            pooled = z.mean(dim=1)   # (B, D)

        out = self.head(pooled)      # (B, 1)
        return out

In [27]:
# ============================================================
# 2) Modelo: factory configurable para tuning
# ============================================================
import torch.nn as nn

def make_transformer_model(
    *,
    n_features: int,
    device: torch.device,
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 128,
    dropout: float = 0.1,
    pooling: str = "mean",
) -> nn.Module:
    """
    Factory del Transformer many-to-one.

    Permite variar hiperparámetros desde Optuna o grid search.
    """

    model = TransformerManyToOne(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_ff=dim_ff,
        dropout=dropout,
        pooling=pooling,
    ).to(device)

    return model

In [28]:
# ------------------------------------------------------------
# Modelos independientes (H60 y H90)
# IMPORTANTE: n_features debe coincidir con tu X 3D: (B, L, F)
# ------------------------------------------------------------
n_features = 36

In [29]:
# ------------------------------------------------------------
# Smoke test robusto (por split/horizonte)
# ------------------------------------------------------------
def smoke_test(model, loader, device, name: str):
    """
    Verifica rápidamente:
      - shapes esperados: X (B,L,F), y (B,) o (B,1), out (B,1)
      - dtypes
      - alineación batch entre y y out
      - loader no vacío
    """
    model.eval()

    try:
        xb, yb = next(iter(loader))
    except StopIteration:
        raise ValueError(f"[{name}] Loader vacío: no hay batches para validar.")

    xb = xb.to(device)
    yb = yb.to(device)

    with torch.no_grad():
        out = model(xb)

    print(f"[{name}] X:", tuple(xb.shape), xb.dtype)
    print(f"[{name}] y:", tuple(yb.shape), yb.dtype)
    print(f"[{name}] out:", tuple(out.shape), out.dtype)

    # --- checks estructurales ---
    assert xb.ndim == 3, f"[{name}] X debe ser 3D: (B, L, F). Recibido {xb.shape}"
    assert out.ndim == 2 and out.shape[1] == 1, f"[{name}] Salida debe ser (B, 1). Recibido {out.shape}"
    assert yb.ndim in (1, 2), f"[{name}] y debe ser (B,) o (B,1). Recibido {yb.shape}"

    # --- alineación batch ---
    B = xb.shape[0]
    assert out.shape[0] == B, f"[{name}] Batch mismatch: X B={B}, out B={out.shape[0]}"

    if yb.ndim == 1:
        assert yb.shape[0] == B, f"[{name}] Batch mismatch: y B={yb.shape[0]}, esperado {B}"
    else:  # yb.ndim == 2
        assert yb.shape == (B, 1), f"[{name}] y 2D debe ser (B,1). Recibido {yb.shape}"

### **10.5. Definición de loss, optimizer y funciones de train / eval (sin tqdm)**

In [30]:
# ============================================================
# loss, optimizer y funciones de entrenamiento / evaluación
# (preparado para múltiples targets / horizontes / window_size)
# ============================================================

import torch
import torch.nn as nn
from typing import Optional


# ------------------------------------------------------------
# Factory: Loss (regresión)
# ------------------------------------------------------------
def make_criterion() -> nn.Module:
    # MSE para entrenamiento (estable y estándar en regresión)
    return nn.MSELoss()


# ------------------------------------------------------------
# Factory: Optimizer (uno por corrida/modelo)
# ------------------------------------------------------------
def make_optimizer(
    model: nn.Module,
    *,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
) -> torch.optim.Optimizer:
    # AdamW suele funcionar bien con Transformers
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


# ------------------------------------------------------------
# Función de entrenamiento (1 epoch)
# ------------------------------------------------------------
def train_one_epoch(
    model: nn.Module,
    loader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
    *,
    clip_grad_norm: Optional[float] = None,
    use_amp: bool = True,   # opcional: AMP en CUDA
) -> float:
    model.train()
    total_loss = 0.0
    n_samples = 0

    # AMP: solo si hay CUDA disponible y está habilitado
    amp_enabled = bool(use_amp and device.type == "cuda")
    #scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)


    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # 1) Normalización segura de shape de y -> (B,1)
        if yb.ndim == 1:
            yb = yb.unsqueeze(1)

        optimizer.zero_grad(set_to_none=True)

        # 2) Forward + loss (AMP opcional)
        #with torch.cuda.amp.autocast(enabled=amp_enabled):
        with torch.amp.autocast("cuda", enabled=amp_enabled):
            y_hat = model(xb)
            loss = criterion(y_hat, yb)

        # 3) Backward + step (AMP opcional)
        scaler.scale(loss).backward()

        if clip_grad_norm is not None and clip_grad_norm > 0:
            # Si usa AMP, conviene "unscale" antes del clip
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)

        scaler.step(optimizer)
        scaler.update()

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n_samples += bs

    return total_loss / max(n_samples, 1)


# ------------------------------------------------------------
# Función de evaluación (1 epoch)
# ------------------------------------------------------------
@torch.no_grad()
def eval_one_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    device: torch.device,
    *,
    use_amp: bool = True,   # opcional: AMP en CUDA
) -> float:
    model.eval()
    total_loss = 0.0
    n_samples = 0

    amp_enabled = bool(use_amp and device.type == "cuda")

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        # Normalización segura de shape de y -> (B,1)
        if yb.ndim == 1:
            yb = yb.unsqueeze(1)

        #with torch.cuda.amp.autocast(enabled=amp_enabled):
        with torch.amp.autocast("cuda", enabled=amp_enabled):
            y_hat = model(xb)
            loss = criterion(y_hat, yb)

        bs = xb.size(0)
        total_loss += loss.item() * bs
        n_samples += bs

    return total_loss / max(n_samples, 1)


# ------------------------------------------------------------
# Nota importante (para su estrategia de tuning)
# ------------------------------------------------------------
# - Entrene con MSE (loss).
# - Para elegir mejores HP en VALID, calcule R²/DA por fuera
#   (objective de Optuna), usando predicciones completas en VALID.

### **10.6. Loop de entrenamiento completo con early stopping**



In [31]:
import math
import torch
import torch.nn as nn
from typing import Optional, Dict, Any


def fit_one_run(
    *,
    model: nn.Module,
    train_loader,
    valid_loader,
    device: torch.device,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    save_best: bool = True,
    best_path: Optional[str] = None,
    clip_grad_norm: Optional[float] = None,
    use_amp: bool = True,          # (opcional) si usa AMP en train/eval
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Entrena 1 modelo (1 target/horizonte/window_size) con early stopping en VALID.

    - Optimiza MSE (loss).
    - Early stopping monitorea valid_loss.
    - Si save_best=True, restaura el mejor estado al final.
    """
    criterion = make_criterion()
    optimizer = make_optimizer(model, lr=lr, weight_decay=weight_decay)

    best_val = math.inf
    best_epoch = -1
    patience_left = patience

    history = {"train_loss": [], "valid_loss": []}
    best_state = None

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
            clip_grad_norm=clip_grad_norm,
            use_amp=use_amp,
        )
        val_loss = eval_one_epoch(
            model,
            valid_loader,
            criterion,
            device,
            use_amp=use_amp,
        )

        history["train_loss"].append(float(train_loss))
        history["valid_loss"].append(float(val_loss))

        # mejora real si supera min_delta
        improved = (best_val - val_loss) > min_delta

        if improved:
            best_val = float(val_loss)
            best_epoch = epoch
            patience_left = patience

            if save_best:
                # Copia segura del state_dict a CPU (para restaurar luego sin depender del device)
                best_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}

                # Guardado opcional a disco
                if best_path is not None:
                    torch.save(model.state_dict(), best_path)
        else:
            patience_left -= 1

        if verbose:
            print(
                f"epoch {epoch:02d} | "
                f"train_loss={train_loss:.6f} | "
                f"valid_loss={val_loss:.6f} | "
                f"patience_left={patience_left}"
            )

        if patience_left <= 0:
            if verbose:
                print(f"Early stopping: best_valid_loss={best_val:.6f} at epoch {best_epoch}")
            break

    # Restaurar mejor modelo
    if save_best and best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)
        if verbose:
            print(f"Modelo restaurado: epoch {best_epoch} | best_valid_loss={best_val:.6f}")

    return {
        "best_valid_loss": best_val,
        "best_epoch": best_epoch,
        "epochs_ran": epoch,   # último epoch ejecutado (incluye early stop)
        "history": history,
    }

### **10.7. Predicciones Transformer**


Función de predicción (seq2one) -> y_true, y_pred

In [32]:
import numpy as np
import torch


@torch.no_grad()
def predict_seq2one(
    model,
    loader,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Genera predicciones completas para un loader seq2one.

    Retorna:
        y_true: shape (N,)
        y_pred: shape (N,)
    Normaliza automáticamente shapes (B,) o (B,1).
    """

    model.eval()

    y_true_list = []
    y_pred_list = []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)

        # Forward
        y_hat = model(xb)

        # Pasar a CPU numpy
        y_pred = y_hat.cpu().numpy()
        y_true = yb.cpu().numpy()

        # Normalizar shapes -> (B,)
        if y_pred.ndim == 2 and y_pred.shape[1] == 1:
            y_pred = y_pred.squeeze(1)

        if y_true.ndim == 2 and y_true.shape[1] == 1:
            y_true = y_true.squeeze(1)

        y_pred_list.append(y_pred)
        y_true_list.append(y_true)

    y_true_all = np.concatenate(y_true_list, axis=0)
    y_pred_all = np.concatenate(y_pred_list, axis=0)

    return y_true_all, y_pred_all

In [33]:
import numpy as np

def get_metrics_torch_from_loaders(
    loaders: dict,
    model,
    *,
    device: torch.device,
    predict_loader_fn=predict_seq2one,
    compute_r2: bool = True,
) -> tuple[dict, dict]:
    """
    Calcula métricas VALID y TEST para modelos seq2one usando DataLoaders.

    Requisitos:
      - loaders debe contener keys: 'valid' y 'test'
      - predict_loader_fn(model, loader, device=...) -> (y_true_all, y_pred_all)
      - compute_seq2one_metrics(y_true, y_pred, ...) debe devolver dict con métricas (MAE/RMSE/R2/DA, etc.)
    """

    # Validaciones defensivas (errores más claros)
    if "valid" not in loaders or "test" not in loaders:
        raise KeyError("loaders debe contener las claves 'valid' y 'test'.")

    # -------- VALID --------
    y_true_valid, y_pred_valid = predict_loader_fn(model, loaders["valid"], device=device)
    y_true_valid = np.asarray(y_true_valid).reshape(-1)
    y_pred_valid = np.asarray(y_pred_valid).reshape(-1)

    metrics_valid = compute_seq2one_metrics(y_true_valid, y_pred_valid, compute_r2=compute_r2)

    # -------- TEST --------
    y_true_test, y_pred_test = predict_loader_fn(model, loaders["test"], device=device)
    y_true_test = np.asarray(y_true_test).reshape(-1)
    y_pred_test = np.asarray(y_pred_test).reshape(-1)

    metrics_test = compute_seq2one_metrics(y_true_test, y_pred_test, compute_r2=compute_r2)

    return metrics_valid, metrics_test

### **11 Funciones de intregación**

### **11.1. Función `train_transformer`**

In [34]:
from typing import Any, Dict, Tuple
import torch
import torch.nn as nn

def train_transformer(
    loaders: dict,
    *,
    n_features: int,
    device: torch.device,

    # ---- hiperparámetros Transformer ----
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 128,
    dropout: float = 0.1,
    pooling: str = "mean",

    # ---- optim / early stopping ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    clip_grad_norm: float | None = 1.0,
    use_scheduler: bool = False,   # (por ahora ignorado)
    save_best: bool = True,
    best_path: str | None = None,
    use_amp: bool = True,          # (opcional) AMP si lo habilitó en fit/train/eval
    verbose: bool = True,
) -> Tuple[nn.Module, Dict[str, Any], Dict, Dict]:
    """
    Entrena 1 Transformer (1 target/horizonte/window_size) y devuelve:
      (model, hist, metrics_valid, metrics_test)

    Nota:
      - La seed NO se fija acá. Debe fijarse afuera (set_seed(42) en tuning,
        y set_seed(seed_i) en multi-seed).
    """

    # 0) Validaciones defensivas
    for k in ("train", "valid", "test"):
        if k not in loaders:
            raise KeyError(f"loaders debe contener '{k}'.")

    # 1) Modelo
    model = TransformerManyToOne(
        n_features=n_features,
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_ff=dim_ff,
        dropout=dropout,
        pooling=pooling,
    ).to(device)

    # 2) Fit (early stopping en VALID)
    hist = fit_one_run(
        model=model,
        train_loader=loaders["train"],
        valid_loader=loaders["valid"],
        device=device,
        lr=lr,
        weight_decay=weight_decay,
        max_epochs=max_epochs,
        patience=patience,
        min_delta=min_delta,
        save_best=save_best,
        best_path=best_path,
        clip_grad_norm=clip_grad_norm,
        use_amp=use_amp,
        verbose=verbose,
    )

    # 3) Métricas VALID/TEST usando loaders (R²/DA/MAE/RMSE según su compute_seq2one_metrics)
    metrics_valid, metrics_test = get_metrics_torch_from_loaders(
        {"valid": loaders["valid"], "test": loaders["test"]},
        model,
        device=device,
        predict_loader_fn=predict_seq2one,
        compute_r2=True,
    )

    # (Opcional) gap para criterio final
    # Ejemplo si metrics tiene "R2":
    # gap_r2 = float(metrics_valid.get("R2", np.nan)) - float(metrics_test.get("R2", np.nan))
    # metrics_valid["gap_R2_valid_minus_test"] = gap_r2

    return model, hist, metrics_valid, metrics_test


### **11.2. Función `run_transformer`**

In [35]:
import pandas as pd
import torch
import gc
import time

def _ts():
    return time.strftime("%H:%M:%S")


def run_transformer(
    window_size: int,
    *,
    n_features: int = 36,
    batch_size_train: int = 4096,
    batch_size_pred: int = 32768,

    # ---- hiperparámetros Transformer (DEBEN propagarse al train) ----
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 128,
    dropout: float = 0.1,
    pooling: str = "mean",

    # ---- optim / early stopping ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = False,   # declarado, pero no se usa (queda como placeholder)

    # ---- misc ----
    verbose: bool = True,
):
    """
    Runner de 1 corrida seq2one para Transformer, por window_size (L) y target(s).

    IMPORTANTE (para su estrategia):
      - La seed NO se fija aquí. Debe fijarse afuera:
          set_seed(42)  -> tuning
          set_seed(seed_i) -> multi-seed
      - Este runner devuelve un DF con filas para VALID y TEST + HPs usados.

    Depende de:
      - create_bundles(...)
      - make_loaders_from_bundle_3d(...)
      - train_transformer(...)
      - get_metrics_torch_from_loaders(...)
      - metrics_to_df(...)
      - predict_seq2one(...)
    """
    L = int(window_size)

    if verbose:
        print("\n" + "=" * 80)
        print(
            f"[{_ts()}] TRANSFORMER | SEQ2ONE | WINDOW_SIZE=L{L} | (L,F)=({L},{n_features}) "
            f"| d_model={d_model} | head={nhead} | layers={num_layers} | ff={dim_ff} | do={dropout} "
            f"| lr={lr} | wd={weight_decay}"
        )
        print("=" * 80)

    # Ajuste a su caso actual
    targets = ["delta_60"]

    rows = []
    t_global = time.perf_counter()

    for i, target in enumerate(targets, start=1):
        t_target = time.perf_counter()

        if verbose:
            print(f"\n[{_ts()}] [{i}/{len(targets)}] START target='{target}' | L{L}")

        bundle = None
        loaders = None
        model = None
        hist = None
        metrics_valid = None
        metrics_test = None

        try:
            # -------------------------
            # BUILD BUNDLE (3D)
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [BUILD] Creando bundle (flatten_X=False) ...")
            t0 = time.perf_counter()

            (bundle,) = create_bundles(
                window_size=L,
                targets=[target],
                windows_paths=windows_paths,
                scalers_paths=scalers_paths,
                flatten_X=False,  # Transformer necesita 3D: (n, L, F)
            )

            if verbose:
                dt = time.perf_counter() - t0
                try:
                    xshape = bundle["train"]["X"].shape
                    yshape = bundle["train"]["y"].shape
                    print(f"[{_ts()}]   [BUILD] OK | train X={xshape} y={yshape} | dt={dt:.2f}s")
                except Exception:
                    print(f"[{_ts()}]   [BUILD] OK | dt={dt:.2f}s")

            # -------------------------
            # LOADERS (train/valid/test)
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [LOADERS] Creando DataLoaders (3D) ...")
            t0 = time.perf_counter()

            loaders = make_loaders_from_bundle_3d(
                bundle,
                seq_len=L,
                n_features=n_features,
                batch_size=batch_size_train,
            )

            if verbose:
                dt = time.perf_counter() - t0
                try:
                    ntr = len(loaders["train"].dataset)
                    nva = len(loaders["valid"].dataset)
                    nte = len(loaders["test"].dataset)
                    print(f"[{_ts()}]   [LOADERS] OK | n(train/valid/test)=({ntr}/{nva}/{nte}) | dt={dt:.2f}s")
                except Exception:
                    print(f"[{_ts()}]   [LOADERS] OK | dt={dt:.2f}s")

            # -------------------------
            # TRAIN (USANDO LOS HP PASADOS A run_transformer)
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [TRAIN] Iniciando entrenamiento ...")
            t0 = time.perf_counter()

            model, hist, metrics_valid, metrics_test = train_transformer(
                loaders,
                n_features=n_features,
                device=device,

                # >>> CAMBIO CLAVE: NO HARDCODEAR. Propagar HP desde run_transformer
                d_model=d_model,
                nhead=nhead,
                num_layers=num_layers,
                dim_ff=dim_ff,
                dropout=dropout,
                pooling=pooling,

                lr=lr,
                weight_decay=weight_decay,
                max_epochs=max_epochs,
                patience=patience,
                min_delta=min_delta,
                clip_grad_norm=clip_grad_norm,
                verbose=verbose,
            )

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [TRAIN] FIN entrenamiento | dt={dt:.2f}s")

            # -------------------------
            # LIBERAR TRAIN (memoria)
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [MEM] Liberando bundle['train'] y gc.collect() ...")
            del bundle["train"]
            gc.collect()

            # -------------------------
            # PRED LOADERS (valid/test) con batch grande
            # -------------------------
            # Creamos loaders solo para valid/test con batch_size_pred
            if verbose:
                print(f"[{_ts()}]   [PRED] Preparando loaders valid/test (batch_size_pred={batch_size_pred}) ...")
            t0 = time.perf_counter()

            # Workaround: la función make_loaders_from_bundle_3d requiere 'train'
            pred_bundle = {
                "valid": bundle["valid"],
                "test": bundle["test"],
                "train": {"X": bundle["valid"]["X"][:1], "y": bundle["valid"]["y"][:1]},
            }

            pred_loaders = make_loaders_from_bundle_3d(
                pred_bundle,
                seq_len=L,
                n_features=n_features,
                batch_size=batch_size_pred,
                num_workers=0,  # pred suele ser más estable con 0
            )

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [PRED] Loaders OK | dt={dt:.2f}s")

            # -------------------------
            # METRICS (valid/test)
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [METRICS] Calculando métricas (valid/test) ...")
            t0 = time.perf_counter()

            metrics_valid, metrics_test = get_metrics_torch_from_loaders(
                {"valid": pred_loaders["valid"], "test": pred_loaders["test"]},
                model,
                device=device,
                predict_loader_fn=predict_seq2one,
                compute_r2=True,
            )

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [METRICS] OK (valid/test) | dt={dt:.2f}s")

            # -------------------------
            # DF APPEND
            # -------------------------
            if verbose:
                print(f"[{_ts()}]   [DF] Agregando filas a la tabla ...")
            t0 = time.perf_counter()

            df_v = metrics_to_df(
                metrics_valid,
                model="transformer",
                split="valid",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )

            df_t = metrics_to_df(
                metrics_test,
                model="transformer",
                split="test",
                horizon=bundle["horizon"],
                window_size=bundle["window_size"],
                target=bundle["target"],
            )

            # Añadimos HPs + info de entrenamiento (alineado con fit_one_run)
            for df_ in (df_v, df_t):
                df_["d_model"] = d_model
                df_["nhead"] = nhead
                df_["num_layers"] = num_layers
                df_["dim_ff"] = dim_ff
                df_["dropout"] = dropout
                df_["pooling"] = pooling
                df_["lr"] = lr
                df_["weight_decay"] = weight_decay

                # >>> CAMBIO CLAVE: keys coherentes con fit_one_run()
                if isinstance(hist, dict):
                    df_["best_valid_loss"] = hist.get("best_valid_loss")
                    df_["best_epoch"] = hist.get("best_epoch")
                    df_["epochs_ran"] = hist.get("epochs_ran")

            rows.append(df_v)
            rows.append(df_t)

            if verbose:
                dt = time.perf_counter() - t0
                print(f"[{_ts()}]   [DF] OK | dt={dt:.2f}s")

            if verbose:
                dt_target = time.perf_counter() - t_target
                print(f"[{_ts()}] [{i}/{len(targets)}] DONE target='{target}' | dt_total={dt_target:.2f}s")

        finally:
            if verbose:
                print(f"[{_ts()}]   [CLEAN] Liberando objetos ...")

            bundle = loaders = model = hist = metrics_valid = metrics_test = None
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # -------------------------
    # FINAL DF
    # -------------------------
    if verbose:
        print(f"\n[{_ts()}] [FINAL] Concatenando resultados ...")
    t0 = time.perf_counter()

    df_transformer_metrics = (
        pd.concat(rows, ignore_index=True)
          .sort_values(["window_size", "target", "split", "horizon_min", "model"])
          .reset_index(drop=True)
    )

    if verbose:
        dt = time.perf_counter() - t0
        dt_all = time.perf_counter() - t_global
        print(f"[{_ts()}] [FINAL] OK | rows={len(df_transformer_metrics)} | dt_concat={dt:.2f}s | dt_total={dt_all:.2f}s")
        print(
            df_transformer_metrics[["window_size", "target", "split", "horizon_min", "model"]]
            .drop_duplicates()
            .to_string(index=False)
        )

    return df_transformer_metrics


### **11.4. Función `run_transformer_incremental`**

In [36]:
from pathlib import Path
import pandas as pd

def run_transformer_incremental(
    *,
    window_sizes: list[int],

    # ---- targets a correr (debe coincidir con lo que realmente entrena run_transformer) ----
    targets: list[str] = ["delta_60"],

    # ---- hiperparámetros Transformer (fijos en esta función; tuning va afuera) ----
    d_model: int = 64,
    nhead: int = 4,
    num_layers: int = 2,
    dim_ff: int = 128,
    dropout: float = 0.1,
    pooling: str = "mean",

    # ---- optim / early stopping ----
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    clip_grad_norm: float = 1.0,
    use_scheduler: bool = False,   # placeholder (no usado actualmente)

    # ---- data ----
    n_features: int = 36,
    batch_size_train: int = 4096,
    batch_size_pred: int = 32768,

    # ---- persistencia ----
    name: str = "transformer",
    metrics_dir: str = "/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics",
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Ejecuta run_transformer() para cada window_size y va acumulando resultados en parquet.

    IMPORTANTE:
      - NO hace tuning. Solo ejecuta con un set fijo de HP.
      - Para tuning con seed fija, usted debe hacer set_seed(42) ANTES de llamar.
      - En multi-seed, set_seed(seed_i) antes de cada corrida.

    Nota:
      - El skip se basa en (model, window_size, target, split) presentes en df_hist.
    """

    # ------------------------------------------------------------
    # Cargar histórico si existe
    # ------------------------------------------------------------
    metrics_dir = Path(metrics_dir)
    metrics_path = metrics_dir / f"seq2one_{name}_metrics.parquet"

    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    expected_splits = {"valid", "test"}
    expected_targets = set(targets)

    dfs_new = []

    for L in window_sizes:
        L = int(L)

        # ---- Skip robusto (solo si ya están los targets y splits que usted pidió) ----
        if not df_hist.empty:
            dfL = df_hist[(df_hist["model"] == name) & (df_hist["window_size"] == L)]
            if not dfL.empty:
                done_targets = set(dfL["target"].unique())
                done_splits = set(dfL["split"].unique())

                if expected_targets.issubset(done_targets) and expected_splits.issubset(done_splits):
                    if verbose:
                        print(f"[SKIP] {name} L={L} ya existe para targets={sorted(expected_targets)}")
                    continue

        # ---- Ejecutar entrenamiento para L ----
        # Nota: run_transformer actualmente define targets internamente.
        # Recomendación: modificar run_transformer para aceptar `targets` como argumento.
        df_L = run_transformer(
            window_size=L,
            n_features=n_features,
            batch_size_train=batch_size_train,
            batch_size_pred=batch_size_pred,

            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            dim_ff=dim_ff,
            dropout=dropout,
            pooling=pooling,

            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            min_delta=min_delta,
            clip_grad_norm=clip_grad_norm,
            use_scheduler=use_scheduler,
            verbose=verbose,
        )

        # Estandarizar nombre del modelo guardado
        df_L["model"] = name
        dfs_new.append(df_L)

        # Actualizar histórico en memoria
        df_hist = df_L.copy() if df_hist.empty else pd.concat([df_hist, df_L], ignore_index=True)

        # Guardar incremental
        save_seq2one_metrics(df_hist, name=name)

    return (
        df_hist.sort_values(["window_size", "target", "split", "horizon_min", "model"])
              .reset_index(drop=True)
    )

# **TUNING Y PRUEBA MULTI SEED.**

## **12. Tuning de Hiperparámetros del Transformer con Optuna**

Para optimizar el modelo Transformer se adopta un enfoque de *hyperparameter tuning* utilizando **Optuna**, una biblioteca de optimización basada en búsqueda adaptativa.

**Estrategia adoptada**

Siguiendo la metodología definida en el proyecto:

1. **Durante el tuning**
   - Se utiliza una **seed fija (42)** para garantizar comparabilidad entre configuraciones.
   - Se optimiza exclusivamente sobre el **split VALID**.
   - La métrica objetivo es **R² en VALID**.
   - Se aplica early stopping para evitar sobreentrenamiento y reducir costo computacional.

2. **Después del tuning**
   - Se seleccionan las mejores configuraciones (top 1 o top 3).
   - Cada configuración se evalúa con **múltiples seeds (10–20)**.
   - Se reportan:
     - R² medio
     - R² desviación estándar
     - Gap VALID–TEST
     
**Justificación del uso de Optuna**

Optuna permite:

- Explorar el espacio de hiperparámetros de forma más eficiente que una grilla tradicional.
- Reducir el número de combinaciones necesarias.
- Priorizar configuraciones prometedoras mediante búsqueda adaptativa.
- Mantener un equilibrio entre rigor metodológico y costo computacional.

Este enfoque es consistente con el pipeline actual y con la evaluación multi-seed ya aplicada en la selección de `window_size`.

In [37]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 13.4 MB/s eta 0:00:00


In [38]:
import math
import numpy as np
import optuna

def objective_transformer_seq2one(
    trial: optuna.Trial,
    *,
    loaders: dict,
    n_features: int,
    device: torch.device,
    # ---- entrenamiento (fijos para tuning) ----
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    clip_grad_norm: float | None = 1.0,
    verbose: bool = False,
) -> float:
    """
    Objective de Optuna para tunear hiperparámetros del Transformer (seq2one).

    Estrategia (alineada al proyecto):
      - Seed fija = 42 durante tuning (comparabilidad entre trials)
      - Selección por VALID (métrica objetivo: R²_valid)
      - Early stopping por valid_loss (MSE) ya implementado en fit_one_run
      - Luego, fuera de Optuna, se hace evaluación multi-seed sobre top configs

    Retorna:
      - R² en VALID (float), para maximizar.
    """

    # ------------------------------------------------------------
    # 0) Seed fija para el tuning (clave para comparabilidad)
    # ------------------------------------------------------------
    set_seed(42)

    # ------------------------------------------------------------
    # 1) Espacio de búsqueda (HP)
    #    Nota: garantizamos compatibilidad d_model % nhead == 0
    # ------------------------------------------------------------
    d_model = trial.suggest_categorical("d_model", [32, 64, 96, 128, 192, 256])
    nhead  = trial.suggest_categorical("nhead",  [2, 4, 8])

    if d_model % nhead != 0:
        # Trial inválido: se prunea para evitar errores y ahorrar cómputo
        raise optuna.TrialPruned(f"Config inválida: d_model={d_model} no divisible por nhead={nhead}")

    num_layers = trial.suggest_int("num_layers", 1, 4)
    dim_ff     = trial.suggest_categorical("dim_ff", [128, 256, 512, 768, 1024])
    dropout    = trial.suggest_float("dropout", 0.0, 0.3)
    pooling    = trial.suggest_categorical("pooling", ["mean", "last"])

    lr          = trial.suggest_float("lr", 1e-4, 3e-3, log=True)
    weight_decay= trial.suggest_float("weight_decay", 1e-6, 3e-3, log=True)

    # ------------------------------------------------------------
    # 2) Entrenar 1 corrida (seed fija) y medir en VALID/TEST
    # ------------------------------------------------------------
    try:
        model, hist, metrics_valid, metrics_test = train_transformer(
            loaders,
            n_features=n_features,
            device=device,

            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            dim_ff=dim_ff,
            dropout=dropout,
            pooling=pooling,

            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            min_delta=min_delta,
            clip_grad_norm=clip_grad_norm,
            save_best=True,
            best_path=None,
            verbose=verbose,
        )
    except RuntimeError as e:
        # Típico: OOM u otro fallo numérico. Prune para continuar.
        raise optuna.TrialPruned(str(e))

    # ------------------------------------------------------------
    # 3) Objective: R² en VALID
    # ------------------------------------------------------------
    r2_valid = metrics_valid.get("R2", None)
    if r2_valid is None or (isinstance(r2_valid, float) and (math.isnan(r2_valid) or math.isinf(r2_valid))):
        raise optuna.TrialPruned("R2_valid inválido o no disponible")

    r2_valid = float(r2_valid)

    # ------------------------------------------------------------
    # 4) Guardar extras del trial (útil para debug/selección top3)
    # ------------------------------------------------------------
    trial.set_user_attr("metrics_valid", metrics_valid)
    trial.set_user_attr("metrics_test", metrics_test)
    trial.set_user_attr("best_valid_loss", hist.get("best_valid_loss") if isinstance(hist, dict) else None)
    trial.set_user_attr("best_epoch", hist.get("best_epoch") if isinstance(hist, dict) else None)
    trial.set_user_attr("epochs_ran", hist.get("epochs_ran") if isinstance(hist, dict) else None)

    return r2_valid

In [39]:
# study = optuna.create_study(direction="maximize")
# study.optimize(lambda t: objective_transformer_seq2one(
#     t, loaders=loaders, n_features=n_features, device=device,
#     max_epochs=30, patience=5, min_delta=0.0, clip_grad_norm=1.0, verbose=False
# ), n_trials=50)

# best_params = study.best_params
# best_value  = study.best_value  # mejor R²_valid

Antes de ejecutar el study.optimize(...) conviene dejar 3 cosas establecidas para que Optuna corra estable y para que el resultado sea reutilizable.

1. Preparar los loaders una sola vez
- Cree bundle y loaders (train/valid/test) fuera del objective y páselos al objective (como ya está). Esto evita recomputar ventanas en cada trial.

2. Definir el “setup” del estudio

- Dirección: maximize (R²_valid).
- Sampler: TPESampler(seed=42) para reproducibilidad del proceso de búsqueda (además de la seed fija del training).
- (Opcional) Pruner para cortar trials malos.

3. Decidir cómo va a guardar resultados
- Guardar study (y/o un DF con study.trials_dataframe()) para recuperar el top 3 y correr multi-seed después.

In [40]:
# =========================
# 0) Preparar BUNDLE + LOADERS (una sola vez)
# =========================

# Elegir el window_size y target del tuning (ejemplo)
L = 180
target = "delta_60"

# Crear bundle 3D (Transformer)
(bundle,) = create_bundles(
    window_size=L,
    targets=[target],
    windows_paths=windows_paths,
    scalers_paths=scalers_paths,
    flatten_X=False,   # <- clave
)

# Crear DataLoaders
loaders = make_loaders_from_bundle_3d(
    bundle,
    seq_len=L,
    n_features=36,      # o su n_features real
    batch_size=4096,
    num_workers=0,      # recomendado para reproducibilidad durante tuning
)

n_features = 36  # para pasar al objective
print("OK loaders:", {k: len(v.dataset) for k, v in loaders.items()})

H60 Train: (327972, 180, 36) (327972,)
H60 Valid: (70228, 180, 36) (70228,)
H60 Test : (70590, 180, 36) (70590,)
Scaler H60: StandardScaler
OK loaders: {'train': 327972, 'valid': 70228, 'test': 70590}


### **12.1. Coarse Tuning**

#### **Aplicación**

In [41]:
# ============================================================
# TUNEO GRUESO (COARSE) con Optuna
# - Menos epochs + menos HPs pesados => mucho más rápido
# - Persistente (SQLite en Drive) + snapshot parquet por trial
# ============================================================

import time
import math
import warnings
from pathlib import Path

import optuna
import torch


# ------------------------------------------------------------
# 0) Silenciar logs/warnings molestos (opcional)
# ------------------------------------------------------------
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings(
    "ignore",
    message="enable_nested_tensor is True, but self.use_nested_tensor is False*",
    category=UserWarning,
)


# ------------------------------------------------------------
# 1) Objective COARSE (usa sus funciones: set_seed + train_transformer)
# ------------------------------------------------------------
def objective_transformer_coarse(
    trial: optuna.Trial,
    *,
    loaders: dict,
    n_features: int,
    device: torch.device,
    max_epochs: int = 15,
    patience: int = 3,
    min_delta: float = 0.0,
    clip_grad_norm: float | None = 1.0,
    verbose: bool = False,
) -> float:
    # Seed fija durante tuning
    set_seed(42)

    # ---- Espacio recortado (evita modelos enormes) ----
    d_model = trial.suggest_categorical("d_model", [64, 128, 192])
    nhead  = trial.suggest_categorical("nhead",  [4, 8])

    # Garantizar compatibilidad
    if d_model % nhead != 0:
        raise optuna.TrialPruned(f"d_model {d_model} no divisible por nhead {nhead}")

    num_layers = trial.suggest_int("num_layers", 1, 3)
    dim_ff     = trial.suggest_categorical("dim_ff", [256, 512])
    dropout    = trial.suggest_float("dropout", 0.0, 0.25)
    pooling    = trial.suggest_categorical("pooling", ["mean", "last"])

    lr          = trial.suggest_float("lr", 2e-4, 2e-3, log=True)
    weight_decay= trial.suggest_float("weight_decay", 1e-6, 3e-4, log=True)

    # ---- Entrenamiento + métricas ----
    try:
        _, hist, metrics_valid, metrics_test = train_transformer(
            loaders,
            n_features=n_features,
            device=device,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            dim_ff=dim_ff,
            dropout=dropout,
            pooling=pooling,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            min_delta=min_delta,
            clip_grad_norm=clip_grad_norm,
            verbose=verbose,
        )
    except RuntimeError as e:
        # OOM u otros fallos => prune
        raise optuna.TrialPruned(str(e))

    r2_valid = float(metrics_valid.get("R2", float("nan")))
    if math.isnan(r2_valid) or math.isinf(r2_valid):
        raise optuna.TrialPruned("R2_valid inválido")

    # Guardar info útil para análisis posterior
    trial.set_user_attr("metrics_valid", metrics_valid)
    trial.set_user_attr("metrics_test", metrics_test)
    if isinstance(hist, dict):
        trial.set_user_attr("best_valid_loss", hist.get("best_valid_loss"))
        trial.set_user_attr("best_epoch", hist.get("best_epoch"))
        trial.set_user_attr("epochs_ran", hist.get("epochs_ran"))

    return r2_valid


# ------------------------------------------------------------
# 2) Persistencia + callbacks (progreso + snapshot parquet)
# ------------------------------------------------------------
study_name = "transformer_coarse_delta60_L180"

base_dir = Path("/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna")
base_dir.mkdir(parents=True, exist_ok=True)

db_path = base_dir / "transformer_tuning.db"
storage_url = f"sqlite:///{db_path}"

snap_path = base_dir / f"{study_name}_trials.parquet"

t0_global = time.perf_counter()

def progress_callback(study: optuna.Study, trial: optuna.Trial) -> None:
    dt_trial = None
    if trial.datetime_start is not None and trial.datetime_complete is not None:
        dt_trial = (trial.datetime_complete - trial.datetime_start).total_seconds()

    elapsed = time.perf_counter() - t0_global
    status = trial.state.name

    best_val = None
    best_num = None
    if study.best_trial is not None and study.best_value is not None:
        best_val = study.best_value
        best_num = study.best_trial.number

    msg = f"[OPTUNA-COARSE] trial={trial.number:03d} | state={status}"
    if trial.value is not None:
        msg += f" | R2_valid={trial.value:.6f}"
    if dt_trial is not None:
        msg += f" | dt={dt_trial:.1f}s"
    msg += f" | elapsed={elapsed/60:.1f}m"
    if best_val is not None:
        msg += f" | best=trial{best_num} R2={best_val:.6f}"
    print(msg)

def snapshot_callback(study: optuna.Study, trial: optuna.Trial) -> None:
    df = study.trials_dataframe()
    df.to_parquet(snap_path, index=False)


# ------------------------------------------------------------
# 3) Crear/reanudar Study COARSE y correr
# ------------------------------------------------------------
sampler = optuna.samplers.TPESampler(seed=42)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0)

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_url,
    load_if_exists=True,
    sampler=sampler,
    pruner=pruner,
)

print("Trials existentes:", len(study.trials))
print("DB:", db_path)
print("Snapshot parquet:", snap_path)

# IMPORTANT:
# - n_trials aquí es el "grueso": 20–25 suele ser suficiente
target_total = 25
remaining = max(target_total - len(study.trials), 0)

study.optimize(
    lambda t: objective_transformer_coarse(
        t,
        loaders=loaders,
        n_features=n_features,
        device=device,
        max_epochs=15,
        patience=3,
        min_delta=0.0,
        clip_grad_norm=1.0,
        verbose=False,
    ),
    n_trials=remaining,
    callbacks=[progress_callback, snapshot_callback],
)

print("best R2_valid (COARSE):", study.best_value)
print("best params (COARSE):", study.best_params)

df_trials = study.trials_dataframe()

Trials existentes: 28
DB: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna/transformer_tuning.db
Snapshot parquet: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna/transformer_coarse_delta60_L180_trials.parquet
best R2_valid (COARSE): 0.6062661714122846
best params (COARSE): {'d_model': 64, 'nhead': 8, 'num_layers': 2, 'dim_ff': 512, 'dropout': 0.06742781559950205, 'pooling': 'last', 'lr': 0.0007051764931395502, 'weight_decay': 1.9073053158018923e-05}


#### **Observaciones de resultados**

In [42]:
df_trials

,number,value,datetime_start,datetime_complete,duration,params_d_model,params_dim_ff,params_dropout,params_lr,params_nhead,params_num_layers,params_pooling,params_weight_decay,user_attrs_best_epoch,user_attrs_best_valid_loss,user_attrs_epochs_ran,user_attrs_metrics_test,user_attrs_metrics_valid,state
0,0,0.410989,2026-02-26 18:07:50.807478,2026-02-26 18:10:36.356655,0 days 00:02:45.549177,128,512,0.150279,0.001866,4,1,mean,0.000115,2.0,1338.244897,5.0,"{'MAE': 40.66285060469575, 'RMSE': 67.51139320...","{'MAE': 23.528551882641914, 'RMSE': 36.5817848...",COMPLETE
1,1,0.587225,2026-02-26 18:10:36.428791,2026-02-26 18:16:27.823785,0 days 00:05:51.394994,64,512,0.034873,0.000572,8,2,last,0.000088,5.0,937.821564,8.0,"{'MAE': 31.450418189914526, 'RMSE': 59.6468249...","{'MAE': 18.28064473765906, 'RMSE': 30.62383276...",COMPLETE
2,2,0.374841,2026-02-26 18:16:27.896433,2026-02-26 18:22:27.206594,0 days 00:05:59.310161,192,512,0.241408,0.000250,8,1,mean,0.000050,5.0,1420.435746,8.0,"{'MAE': 39.60538402541993, 'RMSE': 66.89657631...","{'MAE': 23.87649202954551, 'RMSE': 37.68759701...",COMPLETE
3,3,0.404808,2026-02-26 18:22:27.258439,2026-02-26 18:26:44.223309,0 days 00:04:16.964870,192,256,0.130017,0.001865,8,1,mean,0.000083,3.0,1352.210904,6.0,"{'MAE': 48.14648041900937, 'RMSE': 77.12115032...","{'MAE': 22.4450643491158, 'RMSE': 36.773225972...",COMPLETE
4,4,0.548946,2026-02-26 18:26:44.285238,2026-02-26 18:30:56.359218,0 days 00:04:12.073980,64,512,0.097169,0.000455,4,1,last,0.000005,7.0,1024.921498,10.0,"{'MAE': 32.73058466849423, 'RMSE': 61.04514963...","{'MAE': 19.313712606464815, 'RMSE': 32.0123493...",COMPLETE
5,5,0.564320,2026-02-26 18:30:56.421109,2026-02-26 18:37:30.122917,0 days 00:06:33.701808,192,256,0.203865,0.001181,8,3,last,0.000002,1.0,989.879759,4.0,"{'MAE': 31.320503503407156, 'RMSE': 60.9853242...","{'MAE': 19.35637113118916, 'RMSE': 31.46205733...",COMPLETE
6,6,0.505480,2026-02-26 18:37:30.174221,2026-02-26 18:41:41.030554,0 days 00:04:10.856333,192,512,0.182402,0.000593,4,1,last,0.000002,3.0,1123.612710,6.0,"{'MAE': 41.16567384069747, 'RMSE': 64.75377183...","{'MAE': 21.385832238660765, 'RMSE': 33.5193098...",COMPLETE
7,7,0.557696,2026-02-26 18:41:41.082176,2026-02-26 18:47:08.256615,0 days 00:05:27.174439,128,256,0.026973,0.000412,4,2,last,0.000018,4.0,1004.919869,7.0,"{'MAE': 31.339944425560052, 'RMSE': 61.3909385...","{'MAE': 18.954537942213058, 'RMSE': 31.7002967...",COMPLETE
8,8,0.370084,2026-02-26 18:47:08.319736,2026-02-26 18:49:51.186714,0 days 00:02:42.866978,64,256,0.232424,0.001488,4,1,mean,0.000098,4.0,1431.686320,7.0,"{'MAE': 41.50584825348573, 'RMSE': 67.80286054...","{'MAE': 24.055642068949634, 'RMSE': 37.8306959...",COMPLETE
9,9,NaN,2026-02-26 18:49:51.238370,2026-02-26 18:55:53.897771,0 days 00:06:02.659401,128,512,0.106777,0.000203,8,1,last,0.000018,NaN,NaN,NaN,NaN,NaN,FAIL


1. **El modelo ganador está claramente identificado**  
   El mejor resultado corresponde al Trial 16 con un R²_valid = 0.6063.  
   Los hiperparámetros asociados son:
   - `d_model = 64`  
   - `dim_ff = 512`  
   - `nhead = 8`  
   - `num_layers = 2`  
   - `pooling = last`  
   - `dropout ≈ 0.07`  
   - `lr ≈ 7e-4`  
   - `weight_decay ≈ 2e-5`  
   La arquitectura ganadora es de tamaño pequeño–medio; los modelos más grandes no aportaron mejoras.

2. **Existe un patrón muy consistente en los mejores trials**  
   Entre los trials 14 y 26 se repite prácticamente la misma estructura:
   - `d_model = 64`  
   - `dim_ff = 512`  
   - `nhead = 8`  
   - `num_layers = 2`  
   - `pooling = last`  
   Todos ellos obtienen R²_valid en el rango aproximado 0.599 – 0.606.  
   Esto indica que el espacio óptimo ya fue correctamente identificado en el coarse tuning.

3. **Las arquitecturas más grandes no mejoran el desempeño**  
   Configuraciones con:
   - `d_model = 192`  
   - `num_layers = 3`  
   - `dropout > 0.20`  
   obtuvieron R²_valid en el rango 0.37 – 0.50.  
   Aumentar tamaño y complejidad no solo no mejora, sino que tiende a degradar el rendimiento.

4. **El pooling = "last" domina claramente sobre "mean"**  
   Los mejores resultados utilizan sistemáticamente `pooling = last`.  
   Las configuraciones con `mean` quedaron consistentemente por debajo.  
   Esto sugiere que el último estado temporal contiene mayor información predictiva en este setup.

5. **El rango óptimo de dropout es bajo–medio**  
   Los mejores resultados se concentran entre 0.05 y 0.10.  
   Valores altos (~0.23) deterioran el R²_valid.  
   No se observa necesidad de regularización agresiva.

6. **El learning rate óptimo se encuentra en un rango moderado**  
   Los mejores trials utilizan valores entre aproximadamente 2e-4 y 8e-4.  
   No parece necesario explorar extremos más altos o más bajos.

7. **El proceso coarse ya muestra convergencia estructural**  
   Los mejores resultados:
   - Trial 16 = 0.6063  
   - Trial 24 = 0.6043  
   - Trial 25 = 0.6038  
   - Trial 26 = 0.6036  
   muestran una estabilización del R²_valid alrededor de 0.60 – 0.61.  
   No se observan mejoras significativas adicionales al seguir explorando el mismo espacio amplio.

8. **Conclusión general del coarse tuning**  
   Para el target `delta_60` con `L=180`, la configuración estructural óptima es:
   - Transformer compacto  
   - 2 capas  
   - `d_model = 64`  
   - `dim_ff = 512`  
   - `nhead = 8`  
   - `pooling = last`  
   - `dropout ≈ 0.07`  
   El R²_valid máximo alcanzado se sitúa en torno a 0.60 – 0.61.

9. **Siguiente paso**  
   Realizar un fine tuning centrado en esa arquitectura, ajustando finamente:
   - `dropout` en un rango estrecho (por ejemplo 0.04 – 0.12)  
   - `lr` alrededor del rango identificado (por ejemplo 3e-4 – 9e-4)  
   - `weight_decay` en un rango bajo (por ejemplo 1e-6 – 5e-5)  
   Luego, evaluar la mejor configuración mediante análisis multi-seed para validar robustez.

### **12.2. Fine Tuning**

#### **Aplicación**

In [43]:
# ============================================================
# TUNEO FINO (FINE) con Optuna
# - Arquitectura fija (según coarse)
# - Ajuste fino de lr / weight_decay / dropout
# - Persistente (SQLite en Drive) + snapshot parquet por trial
# ============================================================

import time
import math
import warnings
from pathlib import Path

import optuna
import torch


# ------------------------------------------------------------
# 0) Silenciar logs/warnings (opcional)
# ------------------------------------------------------------
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings(
    "ignore",
    message="enable_nested_tensor is True, but self.use_nested_tensor is False*",
    category=UserWarning,
)


# ------------------------------------------------------------
# 1) Objective FINE (usa: set_seed + train_transformer)
#    - Fija arquitectura ganadora del coarse:
#      d_model=64, nhead=8, num_layers=2, dim_ff=512, pooling=last
#    - Tunea solo: dropout, lr, weight_decay
# ------------------------------------------------------------
def objective_transformer_fine(
    trial: optuna.Trial,
    *,
    loaders: dict,
    n_features: int,
    device: torch.device,
    max_epochs: int = 30,
    patience: int = 5,
    min_delta: float = 0.0,
    clip_grad_norm: float | None = 1.0,
    verbose: bool = False,
) -> float:
    # Seed fija durante tuning
    set_seed(42)

    # ---- Arquitectura fija (según coarse) ----
    d_model    = 64
    nhead      = 8
    num_layers = 2
    dim_ff     = 512
    pooling    = "last"

    # ---- Espacio fino (centrado en lo observado) ----
    # dropout óptimo observado ~0.05–0.10
    dropout = trial.suggest_float("dropout", 0.04, 0.12)

    # lr observado en buenos trials ~2e-4–8e-4 (abrimos un poco)
    lr = trial.suggest_float("lr", 2e-4, 9e-4, log=True)

    # weight_decay observado en buenos trials ~1e-6–3e-5 (abrimos un poco)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 5e-5, log=True)

    # ---- Entrenamiento + métricas ----
    try:
        _, hist, metrics_valid, metrics_test = train_transformer(
            loaders,
            n_features=n_features,
            device=device,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            dim_ff=dim_ff,
            dropout=dropout,
            pooling=pooling,
            lr=lr,
            weight_decay=weight_decay,
            max_epochs=max_epochs,
            patience=patience,
            min_delta=min_delta,
            clip_grad_norm=clip_grad_norm,
            verbose=verbose,
        )
    except RuntimeError as e:
        raise optuna.TrialPruned(str(e))

    r2_valid = float(metrics_valid.get("R2", float("nan")))
    if math.isnan(r2_valid) or math.isinf(r2_valid):
        raise optuna.TrialPruned("R2_valid inválido")

    # Guardar info útil para análisis posterior
    trial.set_user_attr("metrics_valid", metrics_valid)
    trial.set_user_attr("metrics_test", metrics_test)
    if isinstance(hist, dict):
        trial.set_user_attr("best_valid_loss", hist.get("best_valid_loss"))
        trial.set_user_attr("best_epoch", hist.get("best_epoch"))
        trial.set_user_attr("epochs_ran", hist.get("epochs_ran"))

    return r2_valid


# ------------------------------------------------------------
# 2) Persistencia + callbacks (progreso + snapshot parquet)
# ------------------------------------------------------------
study_name = "transformer_fine_delta60_L180"

base_dir = Path("/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna")
base_dir.mkdir(parents=True, exist_ok=True)

db_path = base_dir / "transformer_tuning.db"
storage_url = f"sqlite:///{db_path}"

snap_path = base_dir / f"{study_name}_trials.parquet"

t0_global = time.perf_counter()

def progress_callback(study: optuna.Study, trial: optuna.Trial) -> None:
    dt_trial = None
    if trial.datetime_start is not None and trial.datetime_complete is not None:
        dt_trial = (trial.datetime_complete - trial.datetime_start).total_seconds()

    elapsed = time.perf_counter() - t0_global
    status = trial.state.name

    best_val = None
    best_num = None
    if study.best_trial is not None and study.best_value is not None:
        best_val = study.best_value
        best_num = study.best_trial.number

    msg = f"[OPTUNA-FINE] trial={trial.number:03d} | state={status}"
    if trial.value is not None:
        msg += f" | R2_valid={trial.value:.6f}"
    if dt_trial is not None:
        msg += f" | dt={dt_trial:.1f}s"
    msg += f" | elapsed={elapsed/60:.1f}m"
    if best_val is not None:
        msg += f" | best=trial{best_num} R2={best_val:.6f}"
    print(msg)

def snapshot_callback(study: optuna.Study, trial: optuna.Trial) -> None:
    df = study.trials_dataframe()
    df.to_parquet(snap_path, index=False)


# ------------------------------------------------------------
# 3) Crear/reanudar Study FINE y correr
# ------------------------------------------------------------
sampler = optuna.samplers.TPESampler(seed=42)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0)

study = optuna.create_study(
    direction="maximize",
    study_name=study_name,
    storage=storage_url,
    load_if_exists=True,
    sampler=sampler,
    pruner=pruner,
)

print("Trials existentes:", len(study.trials))
print("DB:", db_path)
print("Snapshot parquet:", snap_path)

# Recomendación: 15–25 trials para fine suele ser suficiente
target_total = 25
remaining = max(target_total - len(study.trials), 0)

study.optimize(
    lambda t: objective_transformer_fine(
        t,
        loaders=loaders,
        n_features=n_features,
        device=device,
        max_epochs=30,
        patience=5,
        min_delta=0.0,
        clip_grad_norm=1.0,
        verbose=False,
    ),
    n_trials=remaining,
    callbacks=[progress_callback, snapshot_callback],
)

print("best R2_valid (FINE):", study.best_value)
print("best params (FINE):", study.best_params)

df_trials = study.trials_dataframe()
#df_trials.sort_values("value", ascending=False).head(10)

Trials existentes: 25
DB: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna/transformer_tuning.db
Snapshot parquet: /content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/optuna/transformer_fine_delta60_L180_trials.parquet
best R2_valid (FINE): 0.6111810647190525
best params (FINE): {'dropout': 0.04464668897345596, 'lr': 0.0007359140478045017, 'weight_decay': 1.050210543674428e-05}


#### **Observaciones de resultados**

In [44]:
df_trials.sort_values("value", ascending=False)

,number,value,datetime_start,datetime_complete,duration,params_dropout,params_lr,params_weight_decay,user_attrs_best_epoch,user_attrs_best_valid_loss,user_attrs_epochs_ran,user_attrs_metrics_test,user_attrs_metrics_valid,state
2,2,0.611181,2026-02-27 16:06:27.509142,2026-02-27 16:13:46.499602,0 days 00:07:18.990460,0.044647,0.000736,0.000011,5.0,883.400510,10.0,"{'MAE': 29.806007412969592, 'RMSE': 58.5453917...","{'MAE': 17.612197576131127, 'RMSE': 29.7219135...",COMPLETE
17,17,0.610127,2026-02-27 18:45:41.619844,2026-02-27 19:00:12.828839,0 days 00:14:31.208995,0.084695,0.000246,0.000022,15.0,885.854998,20.0,"{'MAE': 29.568291012482526, 'RMSE': 58.4841679...","{'MAE': 17.406997524037536, 'RMSE': 29.7621664...",COMPLETE
1,1,0.607663,2026-02-27 15:53:26.326375,2026-02-27 16:06:27.437993,0 days 00:13:01.111618,0.087893,0.000253,0.000002,13.0,891.421759,18.0,"{'MAE': 29.30791656728046, 'RMSE': 58.41063086...","{'MAE': 17.442697027547855, 'RMSE': 29.8560728...",COMPLETE
24,24,0.606884,2026-02-27 20:41:40.915282,2026-02-27 20:54:00.372248,0 days 00:12:19.456966,0.100345,0.000237,0.000032,12.0,893.189110,17.0,"{'MAE': 29.544160404193207, 'RMSE': 58.5384509...","{'MAE': 17.537287892790616, 'RMSE': 29.8856831...",COMPLETE
0,0,0.606752,2026-02-27 15:45:19.089180,2026-02-27 15:53:26.170253,0 days 00:08:07.081073,0.069963,0.000836,0.000018,6.0,893.452435,11.0,"{'MAE': 29.502670989728244, 'RMSE': 57.5497551...","{'MAE': 17.688230439623727, 'RMSE': 29.8907210...",COMPLETE
3,3,0.606427,2026-02-27 16:13:46.572774,2026-02-27 16:26:47.319022,0 days 00:13:00.746248,0.096646,0.000206,0.000044,13.0,894.151171,18.0,"{'MAE': 30.29262290353665, 'RMSE': 59.44327842...","{'MAE': 17.547833824417893, 'RMSE': 29.9030573...",COMPLETE
12,12,0.605727,2026-02-27 17:52:24.300265,2026-02-27 18:06:13.824647,0 days 00:13:49.524382,0.113926,0.000204,0.000015,14.0,895.797265,19.0,"{'MAE': 29.407241404619644, 'RMSE': 58.7906840...","{'MAE': 17.559183565683302, 'RMSE': 29.9296642...",COMPLETE
11,11,0.605359,2026-02-27 17:44:22.936845,2026-02-27 17:52:24.249754,0 days 00:08:01.312909,0.091184,0.000659,0.000005,6.0,896.621316,11.0,"{'MAE': 30.029161978448116, 'RMSE': 60.6539288...","{'MAE': 17.751947098129417, 'RMSE': 29.9436223...",COMPLETE
9,9,0.605196,2026-02-27 17:24:44.595494,2026-02-27 17:34:54.969880,0 days 00:10:10.374386,0.081139,0.000488,0.000001,9.0,897.095907,14.0,"{'MAE': 29.57277289714476, 'RMSE': 59.48833377...","{'MAE': 17.582927753515794, 'RMSE': 29.9497868...",COMPLETE
23,23,0.604529,2026-02-27 20:26:29.517285,2026-02-27 20:41:40.855999,0 days 00:15:11.338714,0.117400,0.000265,0.000009,16.0,898.488327,21.0,"{'MAE': 29.639246855425778, 'RMSE': 59.2240889...","{'MAE': 17.47078607676767, 'RMSE': 29.97509618...",COMPLETE


1. **Mejor resultado y magnitud de mejora frente al coarse**  
   El mejor trial del fine tuning es el Trial 2 con R²_valid = 0.611181.  
   En comparación con el mejor coarse (≈ 0.6063), la mejora es pequeña pero real (≈ +0.005). Esto sugiere que la arquitectura ya estaba muy cerca del óptimo y el fine tuning está ajustando detalles.

2. **Top de desempeño está concentrado en pocos trials**  
   Los mejores valores se ubican en un rango estrecho:  
   - Trial 2: 0.611181  
   - Trial 17: 0.610127  
   - Trial 24: 0.606884  
   - Trial 1: 0.607663  
   - Trial 0: 0.606752  
   Esto indica que el rendimiento máximo se estabiliza alrededor de 0.61 y que no hay saltos grandes al ajustar hiperparámetros finos.

3. **Rango de dropout asociado a mejores resultados**  
   Los mejores trials tienden a usar dropout en el rango aproximado 0.04–0.10:  
   - Trial 2: 0.0446  
   - Trial 17: 0.0847  
   - Trial 1: 0.0879  
   Valores más altos (~0.11–0.12) pueden seguir siendo competitivos (por ejemplo Trial 24 y 23), pero no se ve una ventaja clara por aumentar dropout.

4. **Learning rate: mejores resultados con valores moderados**  
   Los mejores trials se concentran en lr entre ~2e-4 y ~8e-4:  
   - Trial 2: 7.36e-4  
   - Trial 17: 2.46e-4  
   - Trial 1: 2.53e-4  
   - Trial 0: 8.36e-4  
   Esto confirma que el rango definido para fine tuning fue adecuado y que no se observa necesidad de explorar valores más extremos.

5. **Weight decay: resultados competitivos con regularización baja**  
   Los mejores trials tienden a weight_decay bajo (orden 1e-6 a 2e-5):  
   - Trial 1: 2e-6  
   - Trial 2: 1.1e-5  
   - Trial 17: 2.2e-5  
   Se observa que weight_decay más alto (~4e-5) no aporta mejoras (por ejemplo Trial 20 y Trial 24 siguen siendo buenos, pero no los mejores).

6. **Early stopping: los mejores suelen entrenar más epochs**  
   Los mejores trials tienden a correr más epochs antes de detenerse:  
   - Trial 1: best_epoch=13, epochs_ran=18  
   - Trial 17: best_epoch=15, epochs_ran=20  
   - Trial 24: best_epoch=12, epochs_ran=17  
   Esto sugiere que, con estos hiperparámetros, el modelo sigue mejorando durante más tiempo y el early stopping está funcionando como mecanismo de control (sin cortar demasiado pronto).

7. **Métricas (VALID) coherentes con la mejora en R²**  
   En los mejores trials, MAE_valid se mantiene alrededor de 17.4–17.7 y RMSE_valid cerca de 29.7–29.9.  
   Esto es consistente con el aumento moderado de R²_valid, es decir, no se observa un salto drástico en error absoluto, sino un ajuste fino en la explicación de varianza.

8. **Comportamiento en TEST: no se aprecia un deterioro marcado en el top**  
   En los mejores trials, las métricas de TEST (MAE/RMSE) se mantienen en rangos similares entre sí (aprox. MAE_test ~29–30 y RMSE_test ~57–59).  
   Esto sugiere que la mejora observada en VALID no está acompañada por un empeoramiento evidente en TEST, aunque la confirmación final debe hacerse con multi-seed.

9. **Trials RUNNING/FAIL: no afectan la conclusión principal**  
   - Trial 19 aparece como RUNNING, por lo que aún puede cambiar el ranking.  
   - No se observan muchos FAIL en la tabla mostrada, lo cual indica que el espacio fino es estable y no está induciendo fallos frecuentes.

10. **Siguiente paso recomendado**  
   Seleccionar top 3 configuraciones del fine tuning (por ejemplo Trials 2, 17 y 1) y ejecutar evaluación con 20 seeds:  
   - Reportar media y desviación estándar de R²_valid y R²_test.  
   - Elegir la configuración final maximizando R² medio y minimizando la variabilidad, manteniendo controlado el gap VALID–TEST.

## **13. Evaluación multi-seed**

### **13.1. Extraer Top-3 configuraciones (por R²_valid)**

In [45]:
df_trials.sort_values("value", ascending=False).head(3)

,number,value,datetime_start,datetime_complete,duration,params_dropout,params_lr,params_weight_decay,user_attrs_best_epoch,user_attrs_best_valid_loss,user_attrs_epochs_ran,user_attrs_metrics_test,user_attrs_metrics_valid,state
2,2,0.611181,2026-02-27 16:06:27.509142,2026-02-27 16:13:46.499602,0 days 00:07:18.990460,0.044647,0.000736,0.000011,5.0,883.400510,10.0,"{'MAE': 29.806007412969592, 'RMSE': 58.5453917...","{'MAE': 17.612197576131127, 'RMSE': 29.7219135...",COMPLETE
17,17,0.610127,2026-02-27 18:45:41.619844,2026-02-27 19:00:12.828839,0 days 00:14:31.208995,0.084695,0.000246,0.000022,15.0,885.854998,20.0,"{'MAE': 29.568291012482526, 'RMSE': 58.4841679...","{'MAE': 17.406997524037536, 'RMSE': 29.7621664...",COMPLETE
1,1,0.607663,2026-02-27 15:53:26.326375,2026-02-27 16:06:27.437993,0 days 00:13:01.111618,0.087893,0.000253,0.000002,13.0,891.421759,18.0,"{'MAE': 29.30791656728046, 'RMSE': 58.41063086...","{'MAE': 17.442697027547855, 'RMSE': 29.8560728...",COMPLETE


In [46]:
top_k = 3

# 1) Filtrar trials válidos (sin NaN) y ordenar por value desc
df_top = (
    df_trials
    .dropna(subset=["value"])
    .sort_values("value", ascending=False)
    .head(top_k)
    .reset_index(drop=True)
)

# 2) Extraer lista de dicts con params (dropout/lr/weight_decay)
top_params_list = df_top[["params_dropout", "params_lr", "params_weight_decay"]].to_dict(orient="records")

# 3) Mostrar resumen compacto (opcional)
cols_show = [
    "number", "value",
    "params_dropout", "params_lr", "params_weight_decay",
    "user_attrs_best_epoch", "user_attrs_epochs_ran",
]
display(df_top[cols_show])

top_params_list

,number,value,params_dropout,params_lr,params_weight_decay,user_attrs_best_epoch,user_attrs_epochs_ran
0,2,0.611181,0.044647,0.000736,0.000011,5.0,10.0
1,17,0.610127,0.084695,0.000246,0.000022,15.0,20.0
2,1,0.607663,0.087893,0.000253,0.000002,13.0,18.0


[{'params_dropout': 0.04464668897345596,
  'params_lr': 0.0007359140478045017,
  'params_weight_decay': 1.050210543674428e-05},
 {'params_dropout': 0.08469504482348361,
  'params_lr': 0.0002461390746445677,
  'params_weight_decay': 2.194309355738225e-05},
 {'params_dropout': 0.08789267873576292,
  'params_lr': 0.00025289679411762573,
  'params_weight_decay': 1.8408992080552519e-06}]

### **13.2. Evaluación multi-seed (20 seeds) sobre Top-3**

In [47]:
import gc
import time
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================
# MULTI-SEED (20 seeds) para TOP-3 configs del fine tuning
# - Usa loaders ya creados
# - Reusa su train_transformer()
# - Guarda resultados incrementalmente en Drive (parquet)
# ============================================================

def _ts() -> str:
    return time.strftime("%H:%M:%S")


# -------------------------
# Seeds (20)
# -------------------------
seeds = [
    1, 42, 123, 777, 9090, 2024, 2121, 31415, 3333, 4444,
    5555, 6666, 8888, 9999, 1010, 1111, 1212, 1313, 1414, 1515
]

# -------------------------
# Parámetros fijos de arquitectura (según coarse)
# -------------------------
ARCH_FIXED = {
    "d_model": 64,
    "nhead": 8,
    "num_layers": 2,
    "dim_ff": 512,
    "pooling": "last",
}

# -------------------------
# Persistencia incremental (reanudable)
# -------------------------
out_dir = Path("/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/multiseed")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "transformer_fine_top3_multiseed.parquet"

if out_path.exists():
    df_hist = pd.read_parquet(out_path)
else:
    df_hist = pd.DataFrame()

rows = []

print(f"[{_ts()}] [MS] out_path={out_path}")
print(f"[{_ts()}] [MS] hist_rows={len(df_hist)}")

# -------------------------
# Loop TOP-3 configs
# top_params_list debe tener dicts con:
#   {'params_dropout':..., 'params_lr':..., 'params_weight_decay':...}
# o bien {'dropout':..., 'lr':..., 'weight_decay':...}
# -------------------------
for cfg_id, params in enumerate(top_params_list, start=1):

    # Acepta ambos formatos por robustez
    dropout = float(params.get("params_dropout", params.get("dropout")))
    lr = float(params.get("params_lr", params.get("lr")))
    weight_decay = float(params.get("params_weight_decay", params.get("weight_decay")))

    for seed in seeds:
        # ---- Skip si ya existe (reanudar) ----
        if not df_hist.empty:
            mask = (df_hist["cfg_id"] == cfg_id) & (df_hist["seed"] == seed)
            if mask.any():
                print(f"[{_ts()}] [MS] SKIP cfg={cfg_id} seed={seed} (ya existe)")
                continue

        print(f"\n[{_ts()}] [MS] RUN cfg={cfg_id}/{len(top_params_list)} seed={seed} "
              f"| do={dropout:.6f} lr={lr:.6g} wd={weight_decay:.6g}")

        try:
            set_seed(seed)

            model, hist, m_valid, m_test = train_transformer(
                loaders,
                n_features=n_features,
                device=device,

                # arquitectura fija
                d_model=int(ARCH_FIXED["d_model"]),
                nhead=int(ARCH_FIXED["nhead"]),
                num_layers=int(ARCH_FIXED["num_layers"]),
                dim_ff=int(ARCH_FIXED["dim_ff"]),
                pooling=str(ARCH_FIXED["pooling"]),

                # fine params
                dropout=float(dropout),
                lr=float(lr),
                weight_decay=float(weight_decay),

                # training
                max_epochs=30,
                patience=5,
                min_delta=0.0,
                clip_grad_norm=1.0,
                verbose=False,
            )

            row = {
                "cfg_id": cfg_id,
                "seed": seed,

                "R2_valid": float(m_valid.get("R2", np.nan)),
                "R2_test":  float(m_test.get("R2", np.nan)),

                "DA_valid": float(m_valid.get("DA", np.nan)),
                "DA_test":  float(m_test.get("DA", np.nan)),

                "MAE_valid": float(m_valid.get("MAE", np.nan)),
                "MAE_test":  float(m_test.get("MAE", np.nan)),

                "RMSE_valid": float(m_valid.get("RMSE", np.nan)),
                "RMSE_test":  float(m_test.get("RMSE", np.nan)),

                "best_epoch": hist.get("best_epoch") if isinstance(hist, dict) else None,
                "epochs_ran": hist.get("epochs_ran") if isinstance(hist, dict) else None,

                # HPs (fine)
                "hp_dropout": dropout,
                "hp_lr": lr,
                "hp_weight_decay": weight_decay,

                # arquitectura fija (para trazabilidad)
                "hp_d_model": ARCH_FIXED["d_model"],
                "hp_nhead": ARCH_FIXED["nhead"],
                "hp_num_layers": ARCH_FIXED["num_layers"],
                "hp_dim_ff": ARCH_FIXED["dim_ff"],
                "hp_pooling": ARCH_FIXED["pooling"],
            }

            rows.append(row)

        finally:
            # limpieza para evitar leaks
            try:
                del model
            except Exception:
                pass
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        # ---- Guardado incremental por corrida (robusto ante cortes) ----
        if rows:
            df_new = pd.DataFrame(rows)
            if df_hist.empty:
                df_hist = df_new.copy()
            else:
                df_hist = pd.concat([df_hist, df_new], ignore_index=True)

            df_hist.to_parquet(out_path, index=False)
            rows = []  # vaciar buffer
            print(f"[{_ts()}] [MS] SAVE rows={len(df_hist)} -> {out_path}")

# DF final en memoria
df_ms = df_hist.sort_values(["cfg_id", "seed"]).reset_index(drop=True)
df_ms.head()

[11:57:56] [MS] out_path=/content/drive/MyDrive/neural_profit/metrics/seq2one_metrics_tuning/multiseed/transformer_fine_top3_multiseed.parquet
[11:57:56] [MS] hist_rows=60
[11:57:56] [MS] SKIP cfg=1 seed=1 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=42 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=123 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=777 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=9090 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=2024 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=2121 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=31415 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=3333 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=4444 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=5555 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=6666 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=8888 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=9999 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=1010 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=1111 (ya existe)
[11:57:56] [MS] SKIP cfg=1 seed=1212 (ya existe)
[

,cfg_id,seed,R2_valid,R2_test,DA_valid,DA_test,MAE_valid,MAE_test,RMSE_valid,RMSE_test,best_epoch,epochs_ran,hp_dropout,hp_lr,hp_weight_decay,hp_d_model,hp_nhead,hp_num_layers,hp_dim_ff,hp_pooling
0,1,1,0.573644,0.464430,0.788927,0.782130,18.576141,30.963385,31.123558,60.467742,5,10,0.044647,0.000736,0.000011,64,8,2,512,last
1,1,42,0.605222,0.480187,0.802500,0.793819,17.510620,30.185562,29.948799,59.571560,6,11,0.044647,0.000736,0.000011,64,8,2,512,last
2,1,123,0.586653,0.428581,0.792427,0.789018,17.784295,29.798635,30.645075,62.458707,6,11,0.044647,0.000736,0.000011,64,8,2,512,last
3,1,777,0.577405,0.484636,0.794228,0.793435,18.140495,29.990172,30.985980,59.316096,5,10,0.044647,0.000736,0.000011,64,8,2,512,last
4,1,1010,0.584692,0.485205,0.785884,0.787428,18.284053,30.366097,30.717681,59.283338,7,12,0.044647,0.000736,0.000011,64,8,2,512,last


In [48]:
df_ms

,cfg_id,seed,R2_valid,R2_test,DA_valid,DA_test,MAE_valid,MAE_test,RMSE_valid,RMSE_test,best_epoch,epochs_ran,hp_dropout,hp_lr,hp_weight_decay,hp_d_model,hp_nhead,hp_num_layers,hp_dim_ff,hp_pooling
0,1,1,0.573644,0.464430,0.788927,0.782130,18.576141,30.963385,31.123558,60.467742,5,10,0.044647,0.000736,0.000011,64,8,2,512,last
1,1,42,0.605222,0.480187,0.802500,0.793819,17.510620,30.185562,29.948799,59.571560,6,11,0.044647,0.000736,0.000011,64,8,2,512,last
2,1,123,0.586653,0.428581,0.792427,0.789018,17.784295,29.798635,30.645075,62.458707,6,11,0.044647,0.000736,0.000011,64,8,2,512,last
3,1,777,0.577405,0.484636,0.794228,0.793435,18.140495,29.990172,30.985980,59.316096,5,10,0.044647,0.000736,0.000011,64,8,2,512,last
4,1,1010,0.584692,0.485205,0.785884,0.787428,18.284053,30.366097,30.717681,59.283338,7,12,0.044647,0.000736,0.000011,64,8,2,512,last
5,1,1111,0.573565,0.449026,0.786112,0.771109,18.551610,31.832726,31.126445,61.331159,6,11,0.044647,0.000736,0.000011,64,8,2,512,last
6,1,1212,0.576310,0.469740,0.783998,0.773921,18.759110,32.086341,31.026110,60.167205,4,9,0.044647,0.000736,0.000011,64,8,2,512,last
7,1,1313,0.589247,0.475603,0.784312,0.779829,18.204304,31.081063,30.548762,59.833690,5,10,0.044647,0.000736,0.000011,64,8,2,512,last
8,1,1414,0.585474,0.476688,0.788013,0.770100,17.959191,31.648697,30.688719,59.771724,8,13,0.044647,0.000736,0.000011,64,8,2,512,last
9,1,1515,0.580160,0.476681,0.777868,0.784275,18.721752,30.031243,30.884814,59.772140,4,9,0.044647,0.000736,0.000011,64,8,2,512,last


1. **Las tres configuraciones son competitivas, pero con diferencias claras en TEST**  
   En VALID, las tres configs suelen moverse en un rango parecido (aprox. 0.56–0.61).  
   La diferencia relevante aparece en TEST: ahí se ve mejor qué configuración generaliza mejor.

2. **cfg_id = 2 muestra el mejor desempeño típico en TEST**  
   La cfg_id=2 (dropout≈0.0847, lr≈2.46e-4, wd≈2.2e-5) alcanza varios de los mejores R²_test (incluyendo valores >0.50) y, en general, mantiene TEST más alto de forma más consistente que cfg_id=1.  
   También mantiene DA_test en torno a ~0.78–0.80 en muchas seeds, lo cual es coherente con una mejora real de generalización.

3. **cfg_id = 1 tiende a ser más “agresiva” y menos estable en TEST**  
   La cfg_id=1 (dropout≈0.0446, lr≈7.36e-4, wd≈1.1e-5) usa un learning rate bastante más alto.  
   En TEST aparecen caídas más marcadas en algunas seeds (por ejemplo valores ~0.40–0.43), lo que sugiere mayor sensibilidad a inicialización y/o mayor riesgo de sobreajuste relativo.

4. **cfg_id = 3 es buena, pero en promedio parece por debajo de cfg_id=2 en TEST**  
   La cfg_id=3 (dropout≈0.0879, lr≈2.53e-4, wd≈2e-6) tiene resultados sólidos y algunos picos en TEST (>0.50), pero también muestra seeds con TEST más bajos (por ejemplo ~0.42).  
   En conjunto, se ve competitiva, aunque no destaca tanto como cfg_id=2.

5. **El gap VALID–TEST existe en todas (normal), pero cfg_id=2 parece controlarlo mejor**  
   En las tres configs el R²_test queda por debajo del R²_valid (esperable).  
   Aun así, cfg_id=2 parece sostener mejor el rendimiento en TEST para un nivel de VALID similar, lo que es exactamente lo que buscamos en la selección final.

6. **Recomendación práctica de selección final**  
   Si su criterio principal es **generalización (R²_test alto y estabilidad por seed)**, la candidata natural para “finalista” es **cfg_id=2**.  
   Como segunda opción razonable, **cfg_id=3**.  
   cfg_id=1 quedaría como alternativa si se prioriza un poco más VALID y se tolera mayor variabilidad.

### **13.4. Resumen final para elegir la mejor config (R²_mean alto + R²_std bajo + gap controlado)**

In [51]:
import numpy as np

# ============================================================
# 1) Agrupar resultados multi-seed por configuración (cfg_id)
#    Cada cfg_id tiene 20 seeds.
#    Aquí calculamos métricas promedio y dispersión.
# ============================================================

summary = (
    df_ms
        # Agrupar todas las filas por configuración
        .groupby("cfg_id")

        # Calcular estadísticas agregadas por grupo
        .agg(
            # --- VALID ---
            R2_valid_mean=("R2_valid", "mean"),  # promedio R2 en VALID
            R2_valid_std =("R2_valid", "std"),   # dispersión (sensibilidad a seed)

            # --- TEST ---
            R2_test_mean =("R2_test", "mean"),   # promedio R2 en TEST (generalización)
            R2_test_std  =("R2_test", "std"),    # estabilidad en TEST

            # --- Directional Accuracy ---
            DA_valid_mean=("DA_valid", "mean"),
            DA_test_mean =("DA_test", "mean"),

            # --- Error absoluto ---
            MAE_valid_mean=("MAE_valid", "mean"),
            MAE_test_mean =("MAE_test", "mean"),
        )

        # Volver a convertir cfg_id en columna normal
        .reset_index()
)

# ============================================================
# 2) Calcular GAP promedio entre VALID y TEST
#    GAP = R2_valid - R2_test
#    Si es grande → posible sobreajuste.
#    Si es pequeño → mejor transferencia a test.
# ============================================================

gap = (
    df_ms
        # Crear columna temporal con el gap por seed
        .assign(gap_valid_minus_test=lambda d: d["R2_valid"] - d["R2_test"])

        # Agrupar por configuración
        .groupby("cfg_id")["gap_valid_minus_test"]

        # Promedio del gap entre las 20 seeds
        .mean()

        # Convertir en DataFrame
        .reset_index(name="gap_valid_minus_test_mean")
)

# ============================================================
# 3) Unir métricas agregadas + gap en un solo DataFrame
# ============================================================

summary = summary.merge(gap, on="cfg_id", how="left")


# ============================================================
# 4) Ordenar según criterio de selección final
#
# Prioridad:
#   1) Mayor R2_test_mean  → mejor generalización
#   2) Menor R2_test_std   → más estabilidad
#   3) Menor gap           → menos sobreajuste
# ============================================================

summary_sorted = summary.sort_values(
    ["R2_test_mean", "R2_test_std", "gap_valid_minus_test_mean"],
    ascending=[False, True, True],  # False = maximizar, True = minimizar
).reset_index(drop=True)

summary_sorted

,cfg_id,R2_valid_mean,R2_valid_std,R2_test_mean,R2_test_std,DA_valid_mean,DA_test_mean,MAE_valid_mean,MAE_test_mean,gap_valid_minus_test_mean
0,2,0.586100,0.009491,0.476808,0.021605,0.791358,0.784349,18.050566,30.365389,0.109291
1,3,0.586208,0.008550,0.475199,0.020677,0.791879,0.783489,18.020130,30.380457,0.111010
2,1,0.582239,0.009785,0.461825,0.025825,0.787274,0.781453,18.306551,30.866381,0.120414


## **14. Conclusión práctica**

1. **La configuración 2 presenta el mejor equilibrio entre rendimiento y estabilidad**

   En promedio, alcanza el mayor `R2_test_mean` y mantiene una dispersión controlada entre seeds, lo que indica que su desempeño no depende de una inicialización particular. Esto es una señal clara de robustez estructural.

2. **Generaliza mejor que las alternativas evaluadas**

   Comparada con las configuraciones 1 y 3, la configuración 2 sostiene de manera más consistente el rendimiento en TEST, con un gap VALID–TEST controlado y sin evidencia de sobreajuste significativo.

3. **Selección final de hiperparámetros**

   Se adopta formalmente la siguiente configuración:

   - `dropout ≈ 0.0847`
   - `lr ≈ 2.46e-4`
   - `weight_decay ≈ 2.2e-5`
   - Arquitectura fija:
     - `d_model = 64`
     - `nhead = 8`
     - `num_layers = 2`
     - `dim_ff = 512`
     - `pooling = "last"`

4. **Cierre metodológico**

   La selección se basa en un proceso completo y formal:

   - Coarse tuning para exploración estructural.
   - Fine tuning para ajuste fino de hiperparámetros.
   - Selección de top-3 configuraciones.
   - Evaluación multi-seed (20 seeds).
   - Análisis de medias, dispersión y gap VALID–TEST.

   En consecuencia, la configuración 2 puede considerarse la versión final robusta del Transformer seq2one para `delta_60` con `L=180`.